In [1]:
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

# !pip install unidecode
from unidecode import unidecode
import os
from tools import *

from sqlalchemy import text
import math

In [2]:
data_path = "./data/raw"

In [3]:
df_fonciere = pd.read_excel(f"{data_path}/valeurs_foncieres.xlsx",)
df_geo = pd.read_excel(f"{data_path}/fr-esr-referentiel-geographique.xlsx")
df_communes = pd.read_excel(f"{data_path}/donnees_communes.xlsx")

# Les dataframes originales
df_fonciere_raw = df_fonciere.copy()
df_geo_raw = df_geo.copy()
df_communes_raw = df_communes.copy()
print("Fichiers sont chargés correctements")

Fichiers sont chargés correctements


In [4]:
print(f"df_fonciere: {df_fonciere.shape}")
print(f"df_geo: {df_geo.shape}")
print(f"df_communes: {df_communes.shape}")

df_fonciere: (34169, 46)
df_geo: (38916, 37)
df_communes: (34991, 9)


# ⚙️PARAMETRES

In [5]:
# Il faut nettoyer la base ou pas?
clean = False

# 🏢VALEURS FONCIERES (34 169, 46)


## 🏷️: Code service, CH Reference document, Articles CGI
**Observation** : Ils sont tous vides

In [6]:
display_stats(
    df_fonciere, columns=["Code service CH", "Reference document", "1 Articles CGI", "2 Articles CGI", "3 Articles CGI", "4 Articles CGI", "5 Articles CGI"]
    )

---

**Colonne: Code service CH**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.26 MB


,Code service CH
count,"0,00"
mean,nan
std,nan
min,nan
25%,nan
50%,nan
75%,nan
max,nan


⚠️ Il y a 34169 valeurs manquantes dans la colonne 'Code service CH' soit 100.0%


...

---

**Colonne: 5 Articles CGI**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.26 MB


,5 Articles CGI
count,"0,00"
mean,nan
std,nan
min,nan
25%,nan
50%,nan
75%,nan
max,nan


⚠️ Il y a 34169 valeurs manquantes dans la colonne '5 Articles CGI' soit 100.0%


## 🏷️: No disposition

In [7]:
df_fonciere['No disposition'].value_counts()

No disposition
1    34093
2       73
3        3
Name: count, dtype: int64

## 🏷️: Valeur fonciere 

In [8]:
if clean:
    df_fonciere = df_fonciere.dropna(subset="Valeur fonciere")

In [9]:
print("Valeur foncière maximum : ")
display(f"{  df_fonciere['Valeur fonciere'].max():,}")
display_stats(df_fonciere, "Valeur fonciere")

Valeur foncière maximum : 


'9,000,000.0'

---

**Colonne: Valeur fonciere**

🔍 Type réel: float64
📊 Mémoire: 0.26 MB


,Valeur fonciere
count,"34_151,00"
mean,"252_847,12"
std,"325_259,36"
min,"537,50"
25%,"104_000,00"
50%,"169_000,00"
75%,"285_000,00"
max,"9_000_000,00"


⚠️ Il y a 18 valeurs manquantes dans la colonne 'Valeur fonciere' soit 0.1%


In [10]:
# Les valeurs négatives ou zéros peuvent biaiser les aggrérations.Il faut avoir NaN à la place de 0.
display('Toutes les valeurs non null sont-elles supérieuers à zéro?  : '
    "✅ OUI" if (df_fonciere["Valeur fonciere"]  > 0).sum() == (df_fonciere["Valeur fonciere"].notna()).sum()
    else "❌ NON"
)

'Toutes les valeurs non null sont-elles supérieuers à zéro?  : ✅ OUI'

## 🏷️: Date mutation

In [11]:
display_stats(df_fonciere, columns="Date mutation")

---

**Colonne: Date mutation**

🔍 Type réel: datetime64[ns]
Période: 2020-01-02 00:00:00 à 2020-06-30 00:00:00
📊 Mémoire: 0.26 MB
Longueur max du texte: 10
Valeurs uniques: 158
Valeurs les plus fréquentes:


,count
Date mutation,
2020-05-29,884
2020-05-20,688
2020-05-15,623
2020-01-31,568
2020-05-28,549


✅ Pas de valeurs manquantes dans la colonne 'Date mutation'


In [12]:
df_fonciere['Date mutation'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 34169 entries, 0 to 34168
Series name: Date mutation
Non-Null Count  Dtype         
--------------  -----         
34169 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 267.1 KB


## 🏷️: No voie

In [13]:
df_fonciere["No voie"] = df_fonciere["No voie"].astype("Int64")
display_stats(df_fonciere, "No voie")

---

**Colonne: No voie**

🔍 Type réel: Int64
📊 Mémoire: 0.29 MB
Longueur max du texte: 4
Valeurs uniques: 1468
Valeurs les plus fréquentes:


,count
No voie,
1,1538
2,1456
3,1246
4,1204
5,1010


⚠️ Il y a 133 valeurs manquantes dans la colonne 'No voie' soit 0.4%


In [14]:
# Valeur négatives ou 0?
(df_fonciere["No voie"] <=0).any()

np.False_

In [15]:
# Display missing values
display(Markdown("</br><b>**************** Missing values ****************</b>"))
df_fonciere.loc[df_fonciere["No voie"].isnull(),["No voie","Type de voie","Code voie","Voie"]]

</br><b>**************** Missing values ****************</b>

,No voie,Type de voie,Code voie,Voie
67,<NA>,NaN,B104,LES SAVELS
293,<NA>,NaN,B030,LES FEUILLATES
330,<NA>,RES,A021,PORT VAUBAN
511,<NA>,RUE,480,PARMENTIER
515,<NA>,NaN,B024,DERRIERE LES TOURNELLES
...,...,...,...,...
31459,<NA>,RES,A034,DU POMMIER
31954,<NA>,RES,A032,LE MONOIKOS
32614,<NA>,CAE,5,D AMONT
32906,<NA>,RUE,18,DES BATELEURS


## 🏷️: B/T/Q

In [16]:
display_stats(df_fonciere, columns=["B/T/Q"])

---

**Colonne: B/T/Q**

🔍 Type réel: object
📊 Mémoire: 1.08 MB
Longueur max du texte: 1
Valeurs uniques: 24
Valeurs les plus fréquentes:


,count
B/T/Q,
B,1424
A,219
T,186
F,149
C,72


⚠️ Il y a 31995 valeurs manquantes dans la colonne 'B/T/Q' soit 93.60000000000001%


In [17]:
# Chercher les valeurs contenant des chiffres
mask = df_fonciere["B/T/Q"].astype(str).str.contains(r"\d", na=False)
df_fonciere_btq_with_digits = df_fonciere[mask]

display(Markdown("**Attention: Il existe aussi des chiffres !!!!!!**"))
print(df_fonciere_btq_with_digits["B/T/Q"].unique())

**Attention: Il existe aussi des chiffres !!!!!!**

[6 2 1 8 3]


In [18]:
# La taille max
print(f"""Taille max :{
    df_fonciere['B/T/Q'].str.len().max()
}"""
)

Taille max :1.0


In [19]:
df_fonciere["B/T/Q"].value_counts().head(5)

B/T/Q
B    1424
A     219
T     186
F     149
C      72
Name: count, dtype: int64

In [20]:
display(Markdown("**Valeurs B/T/Q incohérents**"))
address_cols = ["No voie","B/T/Q","Type de voie","Voie","Code voie","Code type de voie"]
df_fonciere_btq_with_digits[address_cols]

**Valeurs B/T/Q incohérents**

,No voie,B/T/Q,Type de voie,Voie,Code voie,Code type de voie
925,51,6,ALL,DE CAZAUX,135,2
3800,23,2,PL,GUSTAVE COURBET,114,12
9423,2,1,RUE,JEAN BRULEY,2645,0
23430,50,8,RUE,DE METZ,6210,0
27206,2,6,AV,ARISTIDE BRIAND,20,1
30091,9190,3,ALL,ALBERT BOURGES,18,2


## 🏷️: Code type de voie

In [21]:
display_stats(df_fonciere, "Code type de voie")

---

**Colonne: Code type de voie**

🔍 Type réel: int64
📊 Mémoire: 0.26 MB


,Code type de voie
count,"34_169,00"
mean,"4,03"
std,"9,27"
min,"0,00"
25%,"0,00"
50%,"0,00"
75%,"2,00"
max,"79,00"


✅ Pas de valeurs manquantes dans la colonne 'Code type de voie'


In [22]:
mapping_check = df_fonciere.groupby(["Code type de voie"])["Type de voie"].nunique()
if (mapping_check > 1).any():
    print("🚨 Incohérence détectée dans le mapping Code/Type de voie!")
else:
    print("✅ Mapping code/type de voie cohérent.")

✅ Mapping code/type de voie cohérent.


## 🏷️: Type de voie

In [23]:
display_stats(df_fonciere, "Type de voie")

---

**Colonne: Type de voie**

🔍 Type réel: object
📊 Mémoire: 1.67 MB
Longueur max du texte: 4
Valeurs uniques: 79
Valeurs les plus fréquentes:


,count
Type de voie,
RUE,19580
AV,5107
BD,1842
ALL,1262
RES,849


⚠️ Il y a 940 valeurs manquantes dans la colonne 'Type de voie' soit 2.8000000000000003%


In [24]:
mask = df_fonciere['Type de voie'].isna()
df_fonciere.loc[mask, address_cols].head()

,No voie,B/T/Q,Type de voie,Voie,Code voie,Code type de voie
17,822,G,NaN,LA BAYNASSE SUD,B005,8
31,5794,NaN,NaN,GOLF SAINT LAURENT,D346,8
67,<NA>,NaN,NaN,LES SAVELS,B104,8
293,<NA>,NaN,NaN,LES FEUILLATES,B030,8
315,6113,NaN,NaN,SUVERA TORTA,B537,8


## 🏷️: Code voie

In [25]:
display_stats(df_fonciere, "Code voie")

---

**Colonne: Code voie**

🔍 Type réel: object
📊 Mémoire: 1.20 MB
Longueur max du texte: 4
Valeurs uniques: 5765
Valeurs les plus fréquentes:


,count
Code voie,
100,155
20,134
60,130
180,127
40,122


✅ Pas de valeurs manquantes dans la colonne 'Code voie'


In [26]:
# Aucune utilité analytique démontrée pour "Code voie" à ce stade; laissé hors périmètre, sous réserve de documentation ultérieure.
mask = df_fonciere["Code voie"] == 100
df_fonciere.loc[ mask, ["Code voie"]+ address_cols]

,Code voie,No voie,B/T/Q,Type de voie,Voie,Code voie,Code type de voie
613,100,13,NaN,RUE,BAUDIN,100,0
644,100,8,NaN,RUE,DES ANEMONES,100,0
700,100,2,NaN,AV,LE CENTURION,100,1
965,100,1068,NaN,RUE,FAIDHERBE,100,0
1021,100,15,B,AV,DES CARROSSES,100,1
...,...,...,...,...,...,...,...
32842,100,71,NaN,CHE,DE LA PRALAY,100,5
32904,100,12,NaN,RTE,DE FLAUJACQ,100,3
33006,100,134,NaN,RUE,ANTOINE ARNAUD,100,0
33298,100,25,NaN,RUE,DU DOCTEUR LOUIS MARCON,100,0


## 🏷️: Voie

In [27]:
display_stats(df_fonciere, "Voie")

---

**Colonne: Voie**

🔍 Type réel: object
📊 Mémoire: 2.00 MB
Longueur max du texte: 26
Valeurs uniques: 14133
Valeurs les plus fréquentes:


,count
Voie,
DE LA REPUBLIQUE,310
JEAN JAURES,282
DE PARIS,193
VICTOR HUGO,178
GAMBETTA,168


✅ Pas de valeurs manquantes dans la colonne 'Voie'


In [28]:
display(Markdown("**Valeurs incohérentes (On doit pas avoir le type de la voie comme 'RUE' dans la voie)**"))
# Un example avec des données incohérents
mask = df_fonciere["Voie"].str.startswith("RUE", na=False)
df_fonciere.loc[mask, "Voie"]

**Valeurs incohérentes (On doit pas avoir le type de la voie comme 'RUE' dans la voie)**

599       RUE BERTRAND DE VOGUE
1025        RUE JEAN JAURES HLM
22940           RUE DE L ARCEAU
23381       RUE AGATHA CHRISTIE
28448     RUE GEORGES DU MESNIL
29271    RUE ABBE GUILLEMINAULT
29926     RUE FERDINAND BUISSON
30101         RUE CH BAUDELAIRE
Name: Voie, dtype: object

In [29]:
# mask = df_fonciere["Voie"].str.contains(r"\d", na=True)
mask = df_fonciere["Voie"].str.match(r"^\d", na=False)

df_fonciere.loc[mask, address_cols].tail()

,No voie,B/T/Q,Type de voie,Voie,Code voie,Code type de voie
29738,1,NaN,RUE,16E ET 22E DRAGONS,8290,0
30901,129,NaN,RES,129 RUE LAVOISIER,A007,18
31576,1254,NaN,BD,36 E DI DU TEXAS,3010,15
32739,18,NaN,AV,1ERE DIVISION BROSSET,370,1
33833,5,NaN,NaN,5 RUE DE LA BASTERO LES CE,A010,8


In [30]:
df_fonciere["Voie"].apply(type).value_counts()

Voie
<class 'str'>                  34166
<class 'datetime.datetime'>        2
<class 'int'>                      1
Name: count, dtype: int64

In [31]:
display(Markdown("**Valeurs incohérentes (Non texte)**"))
# Un example avec des données incohérents
df_fonciere[~df_fonciere['Voie'].apply(lambda x: isinstance(x, str))]["Voie"]

**Valeurs incohérentes (Non texte)**

10122                    135
16563    1907-06-01 00:00:00
33000    1918-11-11 00:00:00
Name: Voie, dtype: object

## 🏷️: Code ID commune

In [32]:
display_stats(df_fonciere, "Code ID commune")

---

**Colonne: Code ID commune**

🔍 Type réel: int64
📊 Mémoire: 0.26 MB


,Code ID commune
count,"34_169,00"
mean,"2_009,37"
std,"1_051,41"
min,"0,00"
25%,"1_054,00"
50%,"2_288,00"
75%,"3_011,00"
max,"3_214,00"


✅ Pas de valeurs manquantes dans la colonne 'Code ID commune'


In [33]:
df_fonciere[["Code ID commune","Commune"]].head()

,Code ID commune,Commune
0,1,CHEVRY
1,205,ANTIBES
2,142,NICE
3,228,SAINT LAURENT DU VAR
4,326,AUBAGNE


In [34]:
# ATTENTION CE N'EST PAS LE CODE COMMUNE.C'est un id interne
mask = df_fonciere["Commune"] == "LUNEVILLE"
df_fonciere.loc[mask, ["Commune", "Code ID commune"]].head()

,Commune,Code ID commune
729,LUNEVILLE,1558
1261,LUNEVILLE,1558
2406,LUNEVILLE,1558
3351,LUNEVILLE,1558
3352,LUNEVILLE,1558


In [35]:
df_fonciere["Code ID commune"]  = df_fonciere["Code ID commune"].astype(str)
df_fonciere["Code ID commune"].nunique()

3215

In [36]:
# Est-ce que Code ID commune est vraiment unique pour chaque commune?
mask = df_fonciere.groupby(["Code ID commune"])["Commune"].nunique()
df_fonciere.loc[mask[mask > 1], "Commune"]

mapping_check = df_fonciere.groupby(["Code ID commune"])["Commune"].nunique()
if (mapping_check > 1).any():
    print("🚨 Incohérence détectée dans le mapping Code ID commune / Commune !")
else:
    print("✅ Mapping Code ID commune/Commune cohérent.")

✅ Mapping Code ID commune/Commune cohérent.


## 🏷️: Code commune

In [37]:
col = "Code commune"
df_fonciere[col] = df_fonciere[col].apply(lambda x: to_padded_str(x, zero_pad=3))
display_stats(df_fonciere, col)

---

**Colonne: Code commune**

🔍 Type réel: object
📊 Mémoire: 1.69 MB
Longueur max du texte: 3
Valeurs uniques: 666
Valeurs les plus fréquentes:


,count
Code commune,
118,560
109,520
117,479
088,477
063,465


✅ Pas de valeurs manquantes dans la colonne 'Code commune'


In [38]:
df_fonciere["Code commune"].isin(['None', 'nan', 'NaN', '',' ',]).any()

np.False_

## 🏷️: Code postal

In [39]:
# Remplissage de code postal manquant
mask = df_fonciere["Code postal"].isna()
if mask.sum() == 1:
    display(Markdown("**Code postal à corriger.Avant la correction**"))
    display(df_fonciere.loc[mask,["Code postal", "Code departement", "Code commune","Commune"]])
    df_fonciere.loc[mask, "Code postal"] = 20090
    display(Markdown("**Code postal corrigé**"))
    display(df_fonciere.loc[mask,["Code postal", "Code departement", "Code commune","Commune"]])
else:
    print(f"Nombre de lignes à corriger est {mask.sum()}, on ignore la correction !!! ")

**Code postal à corriger.Avant la correction**

,Code postal,Code departement,Code commune,Commune
26834,NaN,2A,004,AJACCIO


**Code postal corrigé**

,Code postal,Code departement,Code commune,Commune
26834,20090.0,2A,004,AJACCIO


In [40]:
df_fonciere["Code postal"] = df_fonciere["Code postal"].apply(lambda x: to_padded_str(x, zero_pad=5))
display_stats(df_fonciere, "Code postal")

---

**Colonne: Code postal**

🔍 Type réel: object
📊 Mémoire: 1.76 MB
Longueur max du texte: 5
Valeurs uniques: 2449
Valeurs les plus fréquentes:


,count
Code postal,
75018,516
75017,470
75015,407
75016,396
75011,383


✅ Pas de valeurs manquantes dans la colonne 'Code postal'


In [41]:
mask = df_fonciere["Commune"] == "AJACCIO"
df_fonciere.loc[mask, ["Code postal", "Code departement", "Code commune","Commune"]].head()

,Code postal,Code departement,Code commune,Commune
908,20090,2A,004,AJACCIO
909,20000,2A,004,AJACCIO
1607,20090,2A,004,AJACCIO
1608,20090,2A,004,AJACCIO
1856,20090,2A,004,AJACCIO


## 🏷️: Commune

In [42]:
display_stats(df_fonciere, "Commune")

---

**Colonne: Commune**

🔍 Type réel: object
📊 Mémoire: 1.94 MB
Longueur max du texte: 30
Valeurs uniques: 3110
Valeurs les plus fréquentes:


,count
Commune,
PARIS 18,516
PARIS 17,470
PARIS 15,407
PARIS 16,394
NICE,393


✅ Pas de valeurs manquantes dans la colonne 'Commune'


In [43]:
# Nom de la commune la plus longue
df_fonciere.loc[df_fonciere["Commune"].str.len().idxmax(),"Commune"]

'ROUFFIGNAC-ST-CERNIN-DE-REILHA'

## 🏷️: Code departement

In [44]:
display_stats(df_fonciere, "Code departement")

---

**Colonne: Code departement**

🔍 Type réel: object
📊 Mémoire: 1.66 MB
Longueur max du texte: 3
Valeurs uniques: 96
Valeurs les plus fréquentes:


,count
Code departement,
75,4843
94,1927
92,1813
78,1586
69,1412


✅ Pas de valeurs manquantes dans la colonne 'Code departement'


In [45]:
display(Markdown("**Codes départements ayant des lettres**"))
# Codes département ayant des lettres
mask = df_fonciere["Code departement"].str.contains(r"[A-Za-z]", na=False)
df_dep_lettres = df_fonciere.loc[mask, :]
display(
    df_dep_lettres.shape,
    pd.DataFrame(df_dep_lettres["Code departement"].value_counts()).head()
)

**Codes départements ayant des lettres**

(251, 46)

,count
Code departement,
2A,246
2B,5


In [46]:
tmp_cols = ["Code ID commune", "Code commune", "Code postal", "Commune", "Code departement"]
display(
    df_dep_lettres.loc[df_dep_lettres["Code departement"] == "2A", tmp_cols].head(3),
    df_dep_lettres.loc[df_dep_lettres["Code departement"] == "2B", tmp_cols].head(3)
)

,Code ID commune,Code commune,Code postal,Commune,Code departement
315,620,362,20144,ZONZA,2A
697,621,041,20169,BONIFACIO,2A
698,628,070,20111,CASAGLIONE,2A


,Code ID commune,Code commune,Code postal,Commune,Code departement
16657,659,020,20220,AREGNO,2B
17139,660,134,20220,L'ILE ROUSSE,2B
18004,662,185,20232,OLETTA,2B


In [47]:
display(Markdown("**Codes département à 3 chiffres**"))
mask = df_fonciere["Code departement"].astype("str").str.len() == 3
display(
    df_fonciere.loc[mask, "Code departement"].shape,
    df_fonciere.loc[mask, "Code departement"].head(3)
)

**Codes département à 3 chiffres**

(195,)

870     972
1121    972
1525    972
Name: Code departement, dtype: object

## 🔑: calc_CODCOM_INSEE

In [48]:
# Différence entre Code commune et Code Id Commune
df_fonciere[["Code ID commune", "Code departement", "Code commune", "Commune"]].head()

,Code ID commune,Code departement,Code commune,Commune
0,1,01,103,CHEVRY
1,205,06,004,ANTIBES
2,142,06,088,NICE
3,228,06,123,SAINT LAURENT DU VAR
4,326,13,005,AUBAGNE


**Observations:**
Code commune INSEE = Code departement + Code Commune

| Commune 		  | Code INSEE       | Code departement | Code Commune  |
|-----------------|------------------|------------------|---------------|
| AUBAGNE         | 13005            | 13               | 005           |
| CHEVRY          | 01103            | 01               | 103           |

In [49]:
df_fonciere["calc_CODCOM_INSEE"] = df_fonciere["Code departement"] + df_fonciere["Code commune"]
mask = df_fonciere["Commune"] == "CHEVRY"
df_fonciere.loc[mask, ["calc_CODCOM_INSEE", "Commune"]]

,calc_CODCOM_INSEE,Commune
0,01103,CHEVRY
4468,01103,CHEVRY
10997,01103,CHEVRY
15261,01103,CHEVRY


In [50]:
tmp_len_cod_com = df_fonciere["calc_CODCOM_INSEE"].astype('str').str.len().max()
if tmp_len_cod_com != 5:
    display(Markdown(f"⚠️ **Attention** : La longuer du code commune INSEE doît être 5! et non pas de {tmp_len_cod_com}"))

⚠️ **Attention** : La longuer du code commune INSEE doît être 5! et non pas de 6

In [51]:
mask = (df_fonciere["calc_CODCOM_INSEE"].str.len() == 6)
display(Markdown(
    f" **Nombre de code commune à 6 chiffres:** {df_fonciere.loc[mask, 'calc_CODCOM_INSEE'].shape[0]}"
))

 **Nombre de code commune à 6 chiffres:** 195

In [52]:
# Où les longeurs de code departement et code communes sont indentiques (3)
mask = (df_fonciere["Code departement"].str.len() == df_fonciere["Code commune"].str.len())
df_fonciere_cc_a_6_chiffres = df_fonciere.loc[mask, ]
temp_cols = ["Code departement", "Code commune", "Commune", "calc_CODCOM_INSEE"]

display(
    df_fonciere_cc_a_6_chiffres.shape,
    df_fonciere_cc_a_6_chiffres[temp_cols].head()
)

(195, 47)

,Code departement,Code commune,Commune,calc_CODCOM_INSEE
870,972,226,SAINTE ANNE,972226
1121,972,229,SCHOELCHER,972229
1525,972,205,CASE PILOTE,972205
1526,972,209,FORT DE FRANCE,972209
1527,972,209,FORT DE FRANCE,972209


In [53]:
#### Correction des code commune INSEE à 6 chiffre en supprimant un chiffre excedentaire de code commune

df_fonciere_cc_a_6_chiffres.loc[:,'calc_CODCOM_INSEE'] = df_fonciere_cc_a_6_chiffres.apply(
                            lambda row: (
                                str(row['Code departement'])[:2] if len(str(row['Code departement'])) == 3 
                            else str(row['Code departement']).zfill(2)) + str(row['Code commune']).zfill(3), 
                            axis=1
)

In [54]:
# Sauvegarde les modifications
df_fonciere_cc_a_6_chiffres.to_csv("./data/extraits/corrections_code_commune_df_fonciere.csv", sep=";")

In [55]:
df_geo["calc_CODCOM_INSEE"] = df_geo["com_id"].str[1:]
tmp_cols = ["com_nom","Commune","calc_CODCOM_INSEE"]
tmp_merge = df_fonciere_cc_a_6_chiffres.merge(df_geo, how='inner', on='calc_CODCOM_INSEE')[tmp_cols]
tmp_merge.loc[tmp_merge["Commune"] != tmp_merge["com_nom"], tmp_cols]

,com_nom,Commune,calc_CODCOM_INSEE
0,Sainte-Anne,SAINTE ANNE,97226
1,SchÅ“lcher,SCHOELCHER,97229
2,Case-Pilote,CASE PILOTE,97205
3,Fort-de-France,FORT DE FRANCE,97209
4,Fort-de-France,FORT DE FRANCE,97209
...,...,...,...
190,Fort-de-France,FORT DE FRANCE,97209
191,Saint-Denis,SAINT DENIS,97411
192,Saint-Denis,SAINT DENIS,97411
193,Le Lamentin,LE LAMENTIN,97213


In [56]:
mask = ( tmp_merge['com_nom'].apply(normalize_commune) != tmp_merge['Commune'].apply(normalize_commune) )
display(
    tmp_merge.loc[mask, tmp_cols].shape,
    tmp_merge.loc[mask, tmp_cols]
)

(0, 3)

,com_nom,Commune,calc_CODCOM_INSEE


In [57]:
# Mets les lignes corrigées dans la dataframe original
shape_before = df_fonciere.shape

df_fonciere.loc[df_fonciere_cc_a_6_chiffres.index,:] = df_fonciere_cc_a_6_chiffres

report_shape_changes(shape_before= shape_before, shape_after=df_fonciere.shape)

Shape avant: (34169, 47)
Shape après: (34169, 47)
  🔄 Aucun changement de dimension


## 🏷️: Préfixe de section

In [58]:
display_stats(df_fonciere, "Préfixe de section")

---

**Colonne: Préfixe de section**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.26 MB


,Préfixe de section
count,"1_143,00"
mean,"736,81"
std,"237,69"
min,"2,00"
25%,"806,00"
50%,"824,00"
75%,"853,00"
max,"911,00"


⚠️ Il y a 33026 valeurs manquantes dans la colonne 'Préfixe de section' soit 96.7%


### Convertir Préfixe de section en string format 'ddd'

In [59]:
# Remplacer NaN par une chaîne vide
df_fonciere["Préfixe de section"] = df_fonciere["Préfixe de section"].fillna('')

# Convertir la colonne en texte, et supprimer les ".0" à la fin
df_fonciere["Préfixe de section"] = df_fonciere["Préfixe de section"].astype(str).str.replace(r'\.0$', '', regex=True)

# S'assurer que chaque code a bien 3 caractères (ex: "1" -> "001")
df_fonciere["Préfixe de section"] = df_fonciere["Préfixe de section"].str.zfill(3)

In [60]:
print(f"Longueur max: {df_fonciere['Préfixe de section'].astype('str').str.len().max()}")
df_fonciere["Préfixe de section"].head()

Longueur max: 3


0    000
1    000
2    000
3    000
4    000
Name: Préfixe de section, dtype: object

## 🏷️: Section

In [61]:
display_stats(df_fonciere, "Section")

---

**Colonne: Section**

🔍 Type réel: object
📊 Mémoire: 1.66 MB
Longueur max du texte: 2
Valeurs uniques: 463
Valeurs les plus fréquentes:


,count
Section,
AB,1389
AC,1250
AE,1159
AD,1103
AI,997


⚠️ Il y a 1 valeurs manquantes dans la colonne 'Section' soit 0.0%


## 🏷️: No plan

In [62]:
display_stats(df_fonciere, "No plan")

---

**Colonne: No plan**

🔍 Type réel: int64
📊 Mémoire: 0.26 MB


,No plan
count,"34_169,00"
mean,"279,60"
std,"462,20"
min,"1,00"
25%,"52,00"
50%,"141,00"
75%,"333,00"
max,"7_910,00"


✅ Pas de valeurs manquantes dans la colonne 'No plan'


In [63]:
df_fonciere["No plan"] = df_fonciere["No plan"].apply(lambda x: to_padded_str(x, zero_pad=2))

In [64]:
display_stats(df_fonciere, "No plan")

---

**Colonne: No plan**

🔍 Type réel: object
📊 Mémoire: 1.68 MB
Longueur max du texte: 4
Valeurs uniques: 1861
Valeurs les plus fréquentes:


,count
No plan,
01,280
02,230
04,222
11,216
05,203


✅ Pas de valeurs manquantes dans la colonne 'No plan'


## 🏷️: No Volume

In [65]:
display_stats(df_fonciere, "No Volume")

---

**Colonne: No Volume**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.26 MB


,No Volume
count,"0,00"
mean,nan
std,nan
min,nan
25%,nan
50%,nan
75%,nan
max,nan


⚠️ Il y a 34169 valeurs manquantes dans la colonne 'No Volume' soit 100.0%


## 🏷️: 1er lot

In [66]:
display_stats(df_fonciere, "1er lot")

---

**Colonne: 1er lot**

🔍 Type réel: object
📊 Mémoire: 1.17 MB
Longueur max du texte: 7
Valeurs uniques: 1555
Valeurs les plus fréquentes:


,count
1er lot,
2,1616
1,1537
3,1282
4,1147
5,892


✅ Pas de valeurs manquantes dans la colonne '1er lot'


In [67]:
df_fonciere['1er lot'].unique()

array([12, 132, 99, ..., 3019, 5031, 1424], shape=(1555,), dtype=object)

In [68]:
# Valeurs contenant des lettres
mask = df_fonciere["1er lot"].str.contains(r"[A-Za-z]", na=False)
df_fonciere.loc[mask, "1er lot"]

2842      1A
5648      1B
11573    75B
19569    23N
22395    26B
Name: 1er lot, dtype: object

## 🏷️: Surface Carrez du 1er lot

In [69]:
display_stats(df_fonciere, "Surface Carrez du 1er lot")

---

**Colonne: Surface Carrez du 1er lot**

🔍 Type réel: float64
📊 Mémoire: 0.26 MB


,Surface Carrez du 1er lot
count,"34_169,00"
mean,"57,64"
std,"53,34"
min,"0,40"
25%,"34,60"
50%,"53,00"
75%,"72,31"
max,"5_153,00"


✅ Pas de valeurs manquantes dans la colonne 'Surface Carrez du 1er lot'


In [70]:
(df_fonciere["Surface Carrez du 1er lot"] <= 0).any()

np.False_

In [71]:
mask = df_fonciere["Type local"] == "Maison"
df_fonciere[mask][["Surface Carrez du 1er lot", "Surface reelle bati"]]

,Surface Carrez du 1er lot,Surface reelle bati
9,105.37,99
11,96.21,100
22,88.45,89
25,48.23,53
70,86.92,91
...,...,...
34133,80.00,84
34140,140.00,122
34143,89.47,89
34144,44.21,42


## 🏷️: Nombre de lots

In [72]:
df_fonciere['Nombre de lots'].unique()

array([ 2,  1,  3,  4,  6,  5,  8,  7,  9, 12, 17, 10, 14, 11])

Sc de 1er lot + Sc de 2eme lot + Sc de 3eme lot + Sc de 4eme lot + Sc de 5eme lot  = ??

In [73]:
surface_columns = [
            'Surface Carrez du 1er lot', 
            'Surface Carrez du 2eme lot', 
            'Surface Carrez du 3eme lot',
            'Surface Carrez du 4eme lot', 
            'Surface Carrez du 5eme lot',
            'Surface reelle bati',
            'Surface terrain',
]

In [74]:
df_fonciere.assign(
    surface_total_des_lots = df_fonciere['Surface Carrez du 1er lot'] 
                            + df_fonciere['Surface Carrez du 2eme lot'] 
                            + df_fonciere['Surface Carrez du 3eme lot'] 
                            + df_fonciere['Surface Carrez du 4eme lot'] 
                            + df_fonciere['Surface Carrez du 5eme lot'] 
)[surface_columns]

,Surface Carrez du 1er lot,Surface Carrez du 2eme lot,Surface Carrez du 3eme lot,Surface Carrez du 4eme lot,Surface Carrez du 5eme lot,Surface reelle bati,Surface terrain
0,48.22,NaN,NaN,NaN,NaN,48,NaN
1,39.11,NaN,NaN,NaN,NaN,40,NaN
2,80.25,NaN,NaN,NaN,NaN,82,NaN
3,27.51,NaN,NaN,NaN,NaN,27,NaN
4,47.33,NaN,NaN,NaN,NaN,47,NaN
...,...,...,...,...,...,...,...
34164,59.50,NaN,NaN,NaN,NaN,63,NaN
34165,55.29,NaN,NaN,NaN,NaN,66,NaN
34166,62.05,NaN,NaN,NaN,NaN,63,NaN
34167,77.88,NaN,NaN,NaN,NaN,77,NaN


In [75]:
mask = ( (df_fonciere["Surface reelle bati"] - df_fonciere['Surface Carrez du 1er lot']) > 100)
df_fonciere.loc[mask, surface_columns].head()

,Surface Carrez du 1er lot,Surface Carrez du 2eme lot,Surface Carrez du 3eme lot,Surface Carrez du 4eme lot,Surface Carrez du 5eme lot,Surface reelle bati,Surface terrain
729,47.60,NaN,NaN,NaN,NaN,157,NaN
1266,44.87,NaN,NaN,NaN,NaN,148,NaN
1923,122.56,NaN,NaN,NaN,NaN,310,NaN
3624,20.55,NaN,NaN,NaN,NaN,289,NaN
3867,9.80,NaN,NaN,NaN,NaN,110,NaN


In [76]:
for col in surface_columns:
    print(f"{col} est-elle entière vide? {'✅' if df_fonciere[col].isna().all else '❌'}")
    

Surface Carrez du 1er lot est-elle entière vide? ✅
Surface Carrez du 2eme lot est-elle entière vide? ✅
Surface Carrez du 3eme lot est-elle entière vide? ✅
Surface Carrez du 4eme lot est-elle entière vide? ✅
Surface Carrez du 5eme lot est-elle entière vide? ✅
Surface reelle bati est-elle entière vide? ✅
Surface terrain est-elle entière vide? ✅


## 🏷️: Code Type local

In [77]:
df_fonciere["Code type local"].unique()

array([2, 1])

## 🏷️: Type local

In [78]:
display_stats(df_fonciere, "Type local")

---

**Colonne: Type local**

🔍 Type réel: object
📊 Mémoire: 1.94 MB
Longueur max du texte: 11
Valeurs uniques: 2
Valeurs les plus fréquentes:


,count
Type local,
Appartement,31378
Maison,2791


✅ Pas de valeurs manquantes dans la colonne 'Type local'


## 🏷️: Identifiant local

In [79]:
display_stats(df_fonciere, "Identifiant local")

---

**Colonne: Identifiant local**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.26 MB


,Identifiant local
count,"0,00"
mean,nan
std,nan
min,nan
25%,nan
50%,nan
75%,nan
max,nan


⚠️ Il y a 34169 valeurs manquantes dans la colonne 'Identifiant local' soit 100.0%


## 🏷️: Surface reelle bati

In [80]:
display_stats(df_fonciere, "Surface reelle bati")

---

**Colonne: Surface reelle bati**

🔍 Type réel: int64
📊 Mémoire: 0.26 MB


,Surface reelle bati
count,"34_169,00"
mean,"56,72"
std,"29,88"
min,"1,00"
25%,"35,00"
50%,"53,00"
75%,"72,00"
max,"379,00"


✅ Pas de valeurs manquantes dans la colonne 'Surface reelle bati'


## 🏷️: Nombre pieces principales

In [81]:
display_stats(df_fonciere, "Nombre pieces principales")

---

**Colonne: Nombre pieces principales**

🔍 Type réel: int64
📊 Mémoire: 0.26 MB


,Nombre pieces principales
count,"34_169,00"
mean,"2,62"
std,"1,22"
min,"0,00"
25%,"2,00"
50%,"3,00"
75%,"3,00"
max,"11,00"


✅ Pas de valeurs manquantes dans la colonne 'Nombre pieces principales'


## 🏷️: Nature culture

In [82]:
display_stats(df_fonciere, "Nature culture")

---

**Colonne: Nature culture**

🔍 Type réel: object
📊 Mémoire: 1.05 MB
Longueur max du texte: 2
Valeurs uniques: 7
Valeurs les plus fréquentes:


,count
Nature culture,
S,211
AG,34
J,4
VI,1
P,1


⚠️ Il y a 33916 valeurs manquantes dans la colonne 'Nature culture' soit 99.3%


## 🏷️: Nature culture special

In [83]:
display_stats(df_fonciere, "Nature culture speciale")

---

**Colonne: Nature culture speciale**

🔍 Type réel: object
📊 Mémoire: 1.04 MB
Longueur max du texte: 5
Valeurs uniques: 4
Valeurs les plus fréquentes:


,count
Nature culture speciale,
PARC,4
IMM,2
VAOC,1
POTAG,1


⚠️ Il y a 34161 valeurs manquantes dans la colonne 'Nature culture speciale' soit 100.0%


In [84]:
mask = df_fonciere["Nature culture"].notna() & df_fonciere["Nature culture speciale"].notna()
df_fonciere.loc[mask, ["Nature culture", "Nature culture speciale"]].head()



,Nature culture,Nature culture speciale
722,AG,IMM
1661,AG,PARC
5249,AG,IMM
11827,VI,VAOC
16233,AG,PARC


## 🏷️: Surface terrain


In [85]:
display_stats(df_fonciere, "Surface terrain")

---

**Colonne: Surface terrain**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.26 MB


,Surface terrain
count,"253,00"
mean,"355,53"
std,"359,07"
min,"1,00"
25%,"115,00"
50%,"271,00"
75%,"470,00"
max,"2_670,00"


⚠️ Il y a 33916 valeurs manquantes dans la colonne 'Surface terrain' soit 99.3%


## 🏷️: Nom de l'acquereur

In [86]:
display_stats(df_fonciere, "Nom de l'acquereur")

---

**Colonne: Nom de l'acquereur**

🔍 Type réel: object
📊 Mémoire: 1.81 MB
Longueur max du texte: 17
Valeurs uniques: 11112
Valeurs les plus fréquentes:


,count
Nom de l'acquereur,
MABROUKI,12
DENOS,11
MOUTTE,10
BALLON,10
GUNDUZ,10


✅ Pas de valeurs manquantes dans la colonne 'Nom de l'acquereur'


## Recheche de clés candidates

In [87]:
_ = get_possible_candidates(df_fonciere, verbose=True)

#### 🔍 Analyse des colonnes pour trouver des clés candidates

> **❌ Aucune clé candidate trouvée**

# 🗺️ REFERENTIEL GEOGRAPHIQUE (38 916, 37)

In [88]:
df_geo.columns

Index(['regrgp_nom', 'reg_nom', 'reg_nom_old', 'aca_nom', 'dep_nom',
       'com_code', 'com_code1', 'com_code2', 'com_id', 'com_nom_maj_court',
       'com_nom_maj', 'com_nom', 'uu_code', 'uu_id', 'uucr_id', 'uucr_nom',
       'ze_id', 'dep_code', 'dep_id', 'dep_nom_num', 'dep_num_nom', 'aca_code',
       'aca_id', 'reg_code', 'reg_id', 'reg_code_old', 'reg_id_old', 'fd_id',
       'fr_id', 'fe_id', 'uu_id_99', 'au_code', 'au_id', 'auc_id', 'auc_nom',
       'uu_id_10', 'geolocalisation', 'calc_CODCOM_INSEE'],
      dtype='object')

In [89]:
commune_cols = [col for col in df_geo.columns if 'com' in col]

## 🔑: calc_CODCOM_INSEE

In [90]:
df_geo["calc_CODCOM_INSEE"] = df_geo["com_id"].str[1:]

## 🏞️ Régions

In [91]:
region_cols = [col for col in df_geo.columns if 'reg' in col]
print(f"Liste des colonnes de régions:\n {region_cols} \n {len(region_cols)} colonnes")

Liste des colonnes de régions:
 ['regrgp_nom', 'reg_nom', 'reg_nom_old', 'reg_code', 'reg_id', 'reg_code_old', 'reg_id_old'] 
 7 colonnes


### 🏷️: regrgp_nom

In [92]:
#Nom du Regroupement de Régions
display_stats(df_geo, "regrgp_nom")

---

**Colonne: regrgp_nom**

🔍 Type réel: object
📊 Mémoire: 2.12 MB
Longueur max du texte: 13
Valeurs uniques: 3
Valeurs les plus fréquentes:


,count
regrgp_nom,
Province,36835
Ile-de-France,1837
DROM-COM,244


✅ Pas de valeurs manquantes dans la colonne 'regrgp_nom'


In [93]:
df_geo["regrgp_nom"].unique()

array(['Province', 'Ile-de-France', 'DROM-COM'], dtype=object)

In [94]:
# Comparaison du nombre de lignes "Ile-de-France" selon la colonne utilisée :
display(
    (df_geo["regrgp_nom"] == "Ile-de-France").sum(),
    (df_geo["reg_nom"] == "Ile-de-France").sum()
)

np.int64(1837)

np.int64(1837)

### 🏷️: reg_nom

**Attention** : "Collectivité d'outre-mer" n'est pas une région!

In [95]:
display_stats(df_geo, "reg_nom")

---

**Colonne: reg_nom**

🔍 Type réel: object
📊 Mémoire: 2.51 MB
Longueur max du texte: 26
Valeurs uniques: 19
Valeurs les plus fréquentes:


,count
reg_nom,
Grand Est,5596
Nouvelle-Aquitaine,4693
Occitanie,4627
Auvergne-Rhône-Alpes,4380
Bourgogne-Franche-Comté,4018


✅ Pas de valeurs manquantes dans la colonne 'reg_nom'


In [96]:
df_geo[df_geo["reg_nom"] == "Collectivités d'outre-mer"].head()

,regrgp_nom,reg_nom,reg_nom_old,aca_nom,dep_nom,com_code,com_code1,com_code2,com_id,com_nom_maj_court,...,fr_id,fe_id,uu_id_99,au_code,au_id,auc_id,auc_nom,uu_id_10,geolocalisation,calc_CODCOM_INSEE
38424,DROM-COM,Collectivités d'outre-mer,Collectivités d'outre-mer,Saint-Pierre-et-Miquelon,Saint-Pierre-et-Miquelon,975501,97501,97501,C97501,MIQUELON LANGLADE,...,FR12,FE1,SO,SO,SO,SO,SO,SO,NaN,97501
38425,DROM-COM,Collectivités d'outre-mer,Collectivités d'outre-mer,Saint-Pierre-et-Miquelon,Saint-Pierre-et-Miquelon,975502,97502,97502,C97502,ST PIERRE,...,FR12,FE1,SO,SO,SO,SO,SO,SO,NaN,97502
38443,DROM-COM,Collectivités d'outre-mer,Collectivités d'outre-mer,Saint-Barthélemy,Saint-Barthélemy,977701,97701,97701,C97701,ST BARTHELEMY,...,FR12,FE1,SO,97701,AU97701,AU97701,Saint-Barthélemy,SO,NaN,97701
38444,DROM-COM,Collectivités d'outre-mer,Collectivités d'outre-mer,Saint-Martin,Saint-Martin,97801,97801,97801,C97801,ST MARTIN,...,FR12,FE1,SO,97801,AU97801,AU97801,Saint-Martin,SO,NaN,97801
38445,DROM-COM,Collectivités d'outre-mer,Collectivités d'outre-mer,Etranger,Terres australes et antarctiques françaises,98403,98403,98403,C98403,ILES EPARSES OCEAN INDIEN,...,FR12,FE1,SO,NaN,SO,SO,"Iles éparses de l'océan indien : Tromelin, Glo...",SO,NaN,98403


In [97]:
df_geo["reg_nom"].value_counts()

reg_nom
Grand Est                     5596
Nouvelle-Aquitaine            4693
Occitanie                     4627
Auvergne-Rhône-Alpes          4380
Bourgogne-Franche-Comté       4018
Hauts-de-France               3952
Normandie                     3385
Centre-Val de Loire           1892
Ile-de-France                 1837
Pays de la Loire              1574
Bretagne                      1320
Provence-Alpes-Côte d'Azur    1032
Corse                          366
Collectivités d'outre-mer       92
Guadeloupe                      34
Mayotte                         34
Martinique                      34
Guyane                          26
La Réunion                      24
Name: count, dtype: int64

### 🏷️: reg_nom_old

**Ancienne nom de régions**

In [98]:
display_stats(df_geo, "reg_nom_old")

---

**Colonne: reg_nom_old**

🔍 Type réel: object
📊 Mémoire: 2.47 MB
Longueur max du texte: 26
Valeurs uniques: 28
Valeurs les plus fréquentes:


,count
reg_nom_old,
Midi-Pyrénées,3053
Rhône-Alpes,3049
Lorraine,2487
Picardie,2373
Aquitaine,2364


✅ Pas de valeurs manquantes dans la colonne 'reg_nom_old'


### 🏷️: reg_code

In [99]:
col = "reg_code"
df_geo[col] = df_geo[col].apply(lambda x: to_padded_str(x, zero_pad=2))
display_stats(df_geo, col)

---

**Colonne: reg_code**

🔍 Type réel: object
📊 Mémoire: 1.89 MB
Longueur max du texte: 2
Valeurs uniques: 19
Valeurs les plus fréquentes:


,count
reg_code,
44,5596
75,4693
76,4627
84,4380
27,4018


✅ Pas de valeurs manquantes dans la colonne 'reg_code'


In [100]:
df_geo["reg_code"].isin(['None', 'nan', 'NaN', '',' ',]).any()

np.False_

In [101]:
df_geo.loc[df_geo["reg_code"] == '04', region_cols].head()

,regrgp_nom,reg_nom,reg_nom_old,reg_code,reg_id,reg_code_old,reg_id_old
38400,DROM-COM,La Réunion,La Réunion,04,R04,4,R04
38401,DROM-COM,La Réunion,La Réunion,04,R04,4,R04
38402,DROM-COM,La Réunion,La Réunion,04,R04,4,R04
38403,DROM-COM,La Réunion,La Réunion,04,R04,4,R04
38404,DROM-COM,La Réunion,La Réunion,04,R04,4,R04


### 🏷️: reg_code_old

In [102]:
df_geo["reg_code_old"] = df_geo["reg_code_old"].astype(str).str.zfill(2)
display_stats(df_geo, "reg_code_old")

---

**Colonne: reg_code_old**

🔍 Type réel: object
📊 Mémoire: 1.89 MB
Longueur max du texte: 2
Valeurs uniques: 28
Valeurs les plus fréquentes:


,count
reg_code_old,
73,3053
82,3049
41,2487
22,2373
72,2364


✅ Pas de valeurs manquantes dans la colonne 'reg_code_old'


### 🏷️: reg_id

In [103]:
display_stats(df_geo, "reg_id")

---

**Colonne: reg_id**

🔍 Type réel: object
📊 Mémoire: 1.93 MB
Longueur max du texte: 3
Valeurs uniques: 19
Valeurs les plus fréquentes:


,count
reg_id,
R44,5596
R75,4693
R76,4627
R84,4380
R27,4018


✅ Pas de valeurs manquantes dans la colonne 'reg_id'


#### reg_id = "R" + reg_code?

In [104]:
_=verify_column_reproducibility(
    df_geo,
    "reg_id",
    lambda df: "R" + df["reg_code"].astype(str).str.zfill(2)
)

✅ **reg_id** peut être calculé à 100%

### 🏷️: reg_id_old

In [105]:
display_stats(df_geo, "reg_id_old")

---

**Colonne: reg_id_old**

🔍 Type réel: object
📊 Mémoire: 1.93 MB
Longueur max du texte: 3
Valeurs uniques: 28
Valeurs les plus fréquentes:


,count
reg_id_old,
R73,3053
R82,3049
R41,2487
R22,2373
R72,2364


✅ Pas de valeurs manquantes dans la colonne 'reg_id_old'


In [106]:
_=verify_column_reproducibility(
    df_geo,
    "reg_id_old",
    lambda df: "R" + df["reg_code_old"].astype(str).str.zfill(2)
)

✅ **reg_id_old** peut être calculé à 100%

## 🎓 Académies

In [107]:
academie_cols = [col for col in df_geo.columns if 'aca' in col]
print(f"Liste des colonnes d'académies:\n {academie_cols} \n {len(academie_cols)} académies")

Liste des colonnes d'académies:
 ['aca_nom', 'aca_code', 'aca_id'] 
 3 académies


In [108]:
mask = df_geo["com_nom_maj"] == "LUNEVILLE"
df_geo.loc[mask, ["com_nom_maj"] + academie_cols ]

,com_nom_maj,aca_nom,aca_code,aca_id
21544,LUNEVILLE,Nancy-Metz,12,A12


### 🏷️: aca_nom

In [109]:
display_stats(df_geo, "aca_nom")

---

**Colonne: aca_nom**

🔍 Type réel: object
📊 Mémoire: 2.19 MB
Longueur max du texte: 24
Valeurs uniques: 37
Valeurs les plus fréquentes:


,count
aca_nom,
Normandie,3385
Toulouse,3053
Nancy-Metz,2487
Amiens,2373
Bordeaux,2364


✅ Pas de valeurs manquantes dans la colonne 'aca_nom'


### 🏷️: aca_code

In [110]:
df_geo["aca_code"] = df_geo["aca_code"].astype(str).str.zfill(2)
display_stats(df_geo, "aca_code")

---

**Colonne: aca_code**

🔍 Type réel: object
📊 Mémoire: 1.89 MB
Longueur max du texte: 3
Valeurs uniques: 37
Valeurs les plus fréquentes:


,count
aca_code,
70,3385
16,3053
12,2487
20,2373
04,2364


✅ Pas de valeurs manquantes dans la colonne 'aca_code'


### 🏷️: aca_id

In [111]:
display_stats(df_geo, "aca_id")

---

**Colonne: aca_id**

🔍 Type réel: object
📊 Mémoire: 1.93 MB
Longueur max du texte: 3
Valeurs uniques: 37
Valeurs les plus fréquentes:


,count
aca_id,
A70,3385
A16,3053
A12,2487
A20,2373
A04,2364


✅ Pas de valeurs manquantes dans la colonne 'aca_id'


In [112]:
_=verify_column_reproducibility(
    df_geo,
    "aca_id",
    lambda df: "A" + df["aca_code"].astype(str).str.zfill(2)
)

❌ **aca_id** ne peut PAS être complètement calculé à 100%<br>🟠 **2 erreurs (0.0%)** sur 38916 lignes

,aca_id,aca_id_check
38443,977,A977
38444,978,A978


## 🛒Départements

In [113]:
departement_cols = [col for col in df_geo.columns if 'dep' in col]
print(f"Liste des colonnes de département:\n {region_cols} \n {len(region_cols)} départements")

Liste des colonnes de département:
 ['regrgp_nom', 'reg_nom', 'reg_nom_old', 'reg_code', 'reg_id', 'reg_code_old', 'reg_id_old'] 
 7 départements


### 🏷️: dep_nom

In [114]:
departement_cols

['dep_nom', 'dep_code', 'dep_id', 'dep_nom_num', 'dep_num_nom']

In [115]:
display_stats(df_geo, "dep_nom")

---

**Colonne: dep_nom**

🔍 Type réel: object
📊 Mémoire: 2.27 MB
Longueur max du texte: 43
Valeurs uniques: 109
Valeurs les plus fréquentes:


,count
dep_nom,
Pas-de-Calais,909
Somme,836
Aisne,834
Moselle,767
Calvados,764


✅ Pas de valeurs manquantes dans la colonne 'dep_nom'


In [116]:
mask = df_geo["com_nom_maj"] == "LUNEVILLE"
df_geo.loc[mask, ["com_nom_maj"] + departement_cols]

,com_nom_maj,dep_nom,dep_code,dep_id,dep_nom_num,dep_num_nom
21544,LUNEVILLE,Meurthe-et-Moselle,54,D054,Meurthe-et-Moselle (54),54 - Meurthe-et-Moselle


### 🏷️: dep_code

In [117]:
col = "dep_code"
df_geo[col] = df_geo[col].apply(lambda x: to_padded_str(x, zero_pad=2))
display_stats(df_geo, col)

---

**Colonne: dep_code**

🔍 Type réel: object
📊 Mémoire: 1.89 MB
Longueur max du texte: 3
Valeurs uniques: 109
Valeurs les plus fréquentes:


,count
dep_code,
62,909
80,836
02,834
57,767
14,764


✅ Pas de valeurs manquantes dans la colonne 'dep_code'


In [118]:
mask = ( df_geo["dep_code"].isin(["01"]) )
df_geo.loc[mask, ["dep_code","com_code", "com_nom"]]

,dep_code,com_code,com_nom
0,01,01001,L'Abergement-Clémenciat
1,01,01002,L'Abergement-de-Varey
2,01,01003,Amareins
3,01,01004,Ambérieu-en-Bugey
4,01,01005,Ambérieux-en-Dombes
...,...,...,...
454,01,01455,Volognat
455,01,01456,Vongnes
456,01,01457,Vonnas
457,01,01458,Vouvray


### 🏷️: dep_id

In [119]:
display_stats(df_geo, "dep_id")

---

**Colonne: dep_id**

🔍 Type réel: object
📊 Mémoire: 1.97 MB
Longueur max du texte: 4
Valeurs uniques: 109
Valeurs les plus fréquentes:


,count
dep_id,
D062,909
D080,836
D002,834
D057,767
D014,764


✅ Pas de valeurs manquantes dans la colonne 'dep_id'


In [120]:
# Est-ce que cet attribut peut être calculé ?
_=verify_column_reproducibility(
    df_geo,
    "dep_id",
    lambda df: "D" + df["dep_code"].astype(str).str.zfill(3)
)

✅ **dep_id** peut être calculé à 100%

### 🏷️: dep_nom_num

In [121]:
display_stats(df_geo, "dep_nom_num")

---

**Colonne: dep_nom_num**

🔍 Type réel: object
📊 Mémoire: 2.45 MB
Longueur max du texte: 49
Valeurs uniques: 109
Valeurs les plus fréquentes:


,count
dep_nom_num,
Pas-de-Calais (62),909
Somme (80),836
Aisne (02),834
Moselle (57),767
Calvados (14),764


✅ Pas de valeurs manquantes dans la colonne 'dep_nom_num'


In [122]:
# Est-ce que cet attribut peut être calculé ?
_ = verify_column_reproducibility(
    df_geo,
    "dep_nom_num",
    lambda df:
        df["dep_nom"].astype(str)
        + " "
        + "("
        + df["dep_code"].astype(str).str.zfill(2)
        + ")"
)

❌ **dep_nom_num** ne peut PAS être complètement calculé à 100%<br>🟠 **49 erreurs (0.1%)** sur 38916 lignes

,dep_nom_num,dep_nom_num_check
38467,Polynésie française (987),Polynésie Française (987)
38468,Polynésie française (987),Polynésie Française (987)
38469,Polynésie française (987),Polynésie Française (987)
38470,Polynésie française (987),Polynésie Française (987)
38471,Polynésie française (987),Polynésie Française (987)


### 🏷️: dep_num_nom

In [123]:
display_stats(df_geo, "dep_num_nom")

---

**Colonne: dep_num_nom**

🔍 Type réel: object
📊 Mémoire: 2.45 MB
Longueur max du texte: 49
Valeurs uniques: 109
Valeurs les plus fréquentes:


,count
dep_num_nom,
62 - Pas-de-Calais,909
80 - Somme,836
02 - Aisne,834
57 - Moselle,767
14 - Calvados,764


✅ Pas de valeurs manquantes dans la colonne 'dep_num_nom'


In [124]:
# Est-ce que cet attribut peut être calculé ?
_=verify_column_reproducibility(
    df_geo,
    "dep_num_nom",
    lambda df:
        df["dep_code"].astype(str).str.zfill(2)
        + " - "
        + df["dep_nom"].astype(str)
)

❌ **dep_num_nom** ne peut PAS être complètement calculé à 100%<br>🟠 **49 erreurs (0.1%)** sur 38916 lignes

,dep_num_nom,dep_num_nom_check
38467,987 - Polynésie française,987 - Polynésie Française
38468,987 - Polynésie française,987 - Polynésie Française
38469,987 - Polynésie française,987 - Polynésie Française
38470,987 - Polynésie française,987 - Polynésie Française
38471,987 - Polynésie française,987 - Polynésie Française


## 🏘️ Communes

In [125]:
mask = df_geo["com_nom_maj"] == "STRASBOURG"
df_geo.loc[mask, commune_cols]

,com_code,com_code1,com_code2,com_id,com_nom_maj_court,com_nom_maj,com_nom
28774,67482,67482,67482,C67482,STRASBOURG,STRASBOURG,Strasbourg


In [126]:
df_geo.columns

Index(['regrgp_nom', 'reg_nom', 'reg_nom_old', 'aca_nom', 'dep_nom',
       'com_code', 'com_code1', 'com_code2', 'com_id', 'com_nom_maj_court',
       'com_nom_maj', 'com_nom', 'uu_code', 'uu_id', 'uucr_id', 'uucr_nom',
       'ze_id', 'dep_code', 'dep_id', 'dep_nom_num', 'dep_num_nom', 'aca_code',
       'aca_id', 'reg_code', 'reg_id', 'reg_code_old', 'reg_id_old', 'fd_id',
       'fr_id', 'fe_id', 'uu_id_99', 'au_code', 'au_id', 'auc_id', 'auc_nom',
       'uu_id_10', 'geolocalisation', 'calc_CODCOM_INSEE'],
      dtype='object')

### 🏷️: com_code

In [127]:
display_stats(df_geo, "com_code")

---

**Colonne: com_code**

🔍 Type réel: object
📊 Mémoire: 2.00 MB
Longueur max du texte: 6
Valeurs uniques: 38916
Valeurs les plus fréquentes:


,count
com_code,
01001,1
01002,1
01003,1
01004,1
01005,1


✅ Pas de valeurs manquantes dans la colonne 'com_code'


- `com_id` = "C" + `com_code` partout ?
    
    NON ❌

In [128]:
incoherent_values = verify_column_reproducibility(
                    df_geo,
                    "com_id",
                    lambda df: "C" + df["com_code"].astype(str),
                    return_df = True
                    )
incoherent_values[["com_id","dep_code", "com_code1", "com_code2","com_code", "com_nom"]]

❌ **com_id** ne peut PAS être complètement calculé à 100%<br>🟠 **138 erreurs (0.4%)** sur 38916 lignes

,com_id,dep_code,com_code1,com_code2,com_code,com_nom
38306,C97101,971,97101,97101,971101,Les Abymes
38307,C97102,971,97102,97102,971102,Anse-Bertrand
38308,C97103,971,97103,97103,971103,Baie-Mahault
38309,C97104,971,97104,97104,971104,Baillif
38310,C97105,971,97105,97105,971105,Basse-Terre
...,...,...,...,...,...,...
38439,C97614,976,97614,97614,976614,Ouangani
38440,C97615,976,97615,97615,976615,Pamandzi
38441,C97616,976,97616,97616,976616,Sada
38442,C97617,976,97617,97617,976617,Tsingoni


In [129]:
# Hypothèse: Les 138 lignes incohérentes correspondent-elles aux codes > 5 caractères ?
# (Un code commune INSEE standard = 5 caractères exactement
df_geo["com_code"].str.len().value_counts()

com_code
5    38778
6      138
Name: count, dtype: int64

In [130]:
display('Peut-on corriger avec com_code1/com_code2 ? (cohérence parfaite requise) : '
    "✅ OUI" if (incoherent_values["com_code1"] == incoherent_values["com_code2"]).all() 
    else "❌ NON"
)

'Peut-on corriger avec com_code1/com_code2 ? (cohérence parfaite requise) : ✅ OUI'

In [131]:
before_shape = df_geo.shape
df_geo.loc[incoherent_values.index, "com_code"] = incoherent_values["com_code1"]
after_shape = df_geo.shape
display(before_shape, after_shape)

(38916, 38)

(38916, 38)

In [132]:
verify_column_reproducibility(
                    df_geo,
                    "com_id",
                    lambda df: "C" + df["com_code"].astype(str),
                    )

✅ **com_id** peut être calculé à 100%

### 🏷️: com_code1

In [133]:
display_stats(df_geo, "com_code1")

---

**Colonne: com_code1**

🔍 Type réel: object
📊 Mémoire: 1.34 MB
Longueur max du texte: 5
Valeurs uniques: 38891
Valeurs les plus fréquentes:


,count
com_code1,
13055,17
69123,10
1006,1
1007,1
1008,1


✅ Pas de valeurs manquantes dans la colonne 'com_code1'


- `com_code1`, `com_code2` sont-ils égaux à `com_code` partout ?
  
  NON ❌

In [134]:
_=verify_column_reproducibility(
    df_geo,
    "com_code2",
    lambda df: df["com_code1"]
)

❌ **com_code2** ne peut PAS être complètement calculé à 100%<br>🟠 **20 erreurs (0.1%)** sur 38916 lignes

,com_code2,com_code2_check
31835,75056,75101
31836,75056,75102
31837,75056,75103
31838,75056,75104
31839,75056,75105


In [135]:
mask = (df_geo["com_code1"] == df_geo["com_code2"])
display(df_geo.loc[~mask, commune_cols].shape)
df_geo.loc[~mask, commune_cols].head()

(20, 7)

,com_code,com_code1,com_code2,com_id,com_nom_maj_court,com_nom_maj,com_nom
31835,75101,75101,75056,C75101,PARIS 1,PARIS 1ER ARRONDISSEMENT,Paris 1er
31836,75102,75102,75056,C75102,PARIS 2,PARIS 2E ARRONDISSEMENT,Paris 2e
31837,75103,75103,75056,C75103,PARIS 3,PARIS 3E ARRONDISSEMENT,Paris 3e
31838,75104,75104,75056,C75104,PARIS 4,PARIS 4E ARRONDISSEMENT,Paris 4e
31839,75105,75105,75056,C75105,PARIS 5,PARIS 5E ARRONDISSEMENT,Paris 5e


In [136]:
# Valeurs non numériques
mask = (~df_geo["com_code1"].astype(str).str.isnumeric()) | (~df_geo["com_code2"].astype(str).str.isnumeric())

df_geo.loc[mask, commune_cols].head()

,com_code,com_code1,com_code2,com_id,com_nom_maj_court,com_nom_maj,com_nom
38550,2A001,2A001,2A001,C2A001,AFA,AFA,Afa
38551,2A004,2A004,2A004,C2A004,AJACCIO,AJACCIO,Ajaccio
38552,2A006,2A006,2A006,C2A006,ALATA,ALATA,Alata
38553,2A008,2A008,2A008,C2A008,ALBITRECCIA,ALBITRECCIA,Albitreccia
38554,2A011,2A011,2A011,C2A011,ALTAGENE,ALTAGENE,Altagène


### 🏷️: com_code2

In [137]:
display_stats(df_geo, "com_code2")

---

**Colonne: com_code2**

🔍 Type réel: object
📊 Mémoire: 1.34 MB
Longueur max du texte: 5
Valeurs uniques: 38871
Valeurs les plus fréquentes:


,count
com_code2,
75056,21
13055,17
69123,10
1002,1
1003,1


✅ Pas de valeurs manquantes dans la colonne 'com_code2'


### 🏷️: com_id

In [138]:
display_stats(df_geo, "com_id")

---

**Colonne: com_id**

🔍 Type réel: object
📊 Mémoire: 2.04 MB
Longueur max du texte: 6
Valeurs uniques: 38916
Valeurs les plus fréquentes:


,count
com_id,
C01001,1
C01002,1
C01003,1
C01004,1
C01005,1


✅ Pas de valeurs manquantes dans la colonne 'com_id'


### 🏷️: com_nom_maj_court

In [139]:
display_stats(df_geo, "com_nom_maj_court")

---

**Colonne: com_nom_maj_court**

🔍 Type réel: object
📊 Mémoire: 2.24 MB
Longueur max du texte: 32
Valeurs uniques: 35572
Valeurs les plus fréquentes:


,count
com_nom_maj_court,
STE COLOMBE,14
ST SAUVEUR,13
STE MARIE,12
BEAULIEU,12
ST SULPICE,11


✅ Pas de valeurs manquantes dans la colonne 'com_nom_maj_court'


- `com_nom_maj_court` peut-il être calculé depuis `com_nom_maj` ? (règle métier claire ?)
  
  NON ❌

In [140]:
_=verify_column_reproducibility(
    df_geo,
    "com_nom_maj_court",
    lambda df: df["com_nom_maj"].str.replace("-"," ").str.replace("'"," ")
)

❌ **com_nom_maj_court** ne peut PAS être complètement calculé à 100%<br>🟠 **5247 erreurs (13.5%)** sur 38916 lignes

,com_nom_maj_court,com_nom_maj_court_check
53,BOURG ST CHRISTOPHE,BOURG SAINT CHRISTOPHE
55,BOYEUX ST JEROME,BOYEUX SAINT JEROME
76,CHALLES,CHALLES LA MONTAGNE
169,GEOVREISSIAT,BEARD GEOVREISSIAT
185,HOSTIAS,HOSTIAZ


### 🏷️: com_nom_maj

In [141]:
display_stats(df_geo, "com_nom_maj")

---

**Colonne: com_nom_maj**

🔍 Type réel: object
📊 Mémoire: 2.25 MB
Longueur max du texte: 32
Valeurs uniques: 35525
Valeurs les plus fréquentes:


,count
com_nom_maj,
SAINTE-COLOMBE,14
SAINT-SAUVEUR,13
SAINTE-MARIE,13
BEAULIEU,12
SAINT-AUBIN,11


✅ Pas de valeurs manquantes dans la colonne 'com_nom_maj'


- Est-ce que `com_nom_maj` = `com_nom`.upper() partout ?

  NON ❌

In [142]:
_=verify_column_reproducibility(
    df_geo,
    "com_nom_maj",
    lambda df: df["com_nom"].apply(lambda x: unidecode(str(x)).upper())
)

❌ **com_nom_maj** ne peut PAS être complètement calculé à 100%<br>🟠 **1049 erreurs (2.7%)** sur 38916 lignes

,com_nom_maj,com_nom_maj_check
14,ARBIGNIEU,ARBOYS EN BUGEY
24,BAGE-LA-VILLE,BAGE-DOMMARTIN
32,BELLEGARDE-SUR-VALSERINE,VALSERHONE
35,BELMONT-LUTHEZIEU,VALROMEY-SUR-SERAN
79,CHAMPDOR,CHAMPDOR-CORCELLES


### 🏷️: com_nom

In [143]:
display_stats(df_geo, "com_nom")

---

**Colonne: com_nom**

🔍 Type réel: object
📊 Mémoire: 2.38 MB
Longueur max du texte: 58
Valeurs uniques: 35872
Valeurs les plus fréquentes:


,count
com_nom,
Sainte-Colombe,13
Saint-Sauveur,12
Beaulieu,12
Sainte-Marie,12
Saint-Loup,11


✅ Pas de valeurs manquantes dans la colonne 'com_nom'


In [144]:
pd.set_option('display.width',100)
df_geo.loc[df_geo["com_nom"].str.len().idxmax(),"com_nom"]

'Terres australes et antarctiques françaises : kerguelen ..'

## 🗺️ Unité urbaine

In [145]:
urban_cols = ['uu_code', 'uu_id', 'uucr_id', 'uucr_nom', 'uu_id_99', 'uu_id_10']

In [146]:
mask = (df_geo["com_nom_maj"] == "LUNEVILLE")
df_geo.loc[mask, urban_cols]

,uu_code,uu_id,uucr_id,uucr_nom,uu_id_99,uu_id_10
21544,54402,UU54402,UU54402,Lunéville,UU54402,UU54403


### 🏷️: uu_code

In [147]:
display_stats(df_geo, "uu_code")

---

**Colonne: uu_code**

🔍 Type réel: object
📊 Mémoire: 1.22 MB
Longueur max du texte: 5
Valeurs uniques: 2467
Valeurs les plus fréquentes:


,count
uu_code,
851,431
760,133
752,94
758,81
33701,73


⚠️ Il y a 31291 valeurs manquantes dans la colonne 'uu_code' soit 80.4%


### 🏷️: uu_id

In [148]:
display_stats(df_geo, "uu_id")

---

**Colonne: uu_id**

🔍 Type réel: object
📊 Mémoire: 1.93 MB
Longueur max du texte: 7
Valeurs uniques: 2468
Valeurs les plus fréquentes:


,count
uu_id,
SO,31291
UU00851,431
UU00760,133
UU00752,94
UU00758,81


✅ Pas de valeurs manquantes dans la colonne 'uu_id'


### 🏷️: uucr_id

In [149]:
display_stats(df_geo, "uucr_id")

---

**Colonne: uucr_id**

🔍 Type réel: object
📊 Mémoire: 2.06 MB
Longueur max du texte: 8
Valeurs uniques: 29946
Valeurs les plus fréquentes:


,count
uucr_id,
SO,3813
UU00851,431
UU00760,133
UU00752,94
UU00758,81


✅ Pas de valeurs manquantes dans la colonne 'uucr_id'


### 🏷️: uucr_nom

In [150]:
display_stats(df_geo, "uucr_nom")

---

**Colonne: uucr_nom**

🔍 Type réel: object
📊 Mémoire: 2.37 MB
Longueur max du texte: 50
Valeurs uniques: 31370
Valeurs les plus fréquentes:


,count
uucr_nom,
Paris,434
Lyon,136
Béthune,94
Toulouse,81
Bordeaux,73


✅ Pas de valeurs manquantes dans la colonne 'uucr_nom'


### 🏷️: uu_id_10

In [151]:
display_stats(df_geo, "uu_id_10")

---

**Colonne: uu_id_10**

🔍 Type réel: object
📊 Mémoire: 1.93 MB
Longueur max du texte: 7
Valeurs uniques: 2294
Valeurs les plus fréquentes:


,count
uu_id_10,
SO,31543
UU00851,432
UU00758,139
UU00752,93
UU31701,73


✅ Pas de valeurs manquantes dans la colonne 'uu_id_10'


### 🏷️: uu_id_99

In [152]:
display_stats(df_geo, "uu_id_99")

---

**Colonne: uu_id_99**

🔍 Type réel: object
📊 Mémoire: 1.92 MB
Longueur max du texte: 7
Valeurs uniques: 2054
Valeurs les plus fréquentes:


,count
uu_id_99,
SO,32834
UU00851,416
UU00757,111
UU31701,72
UU00755,68


✅ Pas de valeurs manquantes dans la colonne 'uu_id_99'


## 🏷️: ze_id

In [153]:
display_stats(df_geo, "ze_id")

---

**Colonne: ze_id**

🔍 Type réel: object
📊 Mémoire: 2.03 MB
Longueur max du texte: 6
Valeurs uniques: 324
Valeurs les plus fréquentes:


,count
ze_id,
SO,3813
ZE0061,715
ZE2210,474
ZE2307,467
ZE7310,455


✅ Pas de valeurs manquantes dans la colonne 'ze_id'


## 🏷️: fd_id

In [154]:
display_stats(df_geo, "fd_id")

---

**Colonne: fd_id**

🔍 Type réel: object
📊 Mémoire: 2.00 MB
Longueur max du texte: 5
Valeurs uniques: 3
Valeurs les plus fréquentes:


,count
fd_id,
FD111,38672
FD112,152
FD121,92


✅ Pas de valeurs manquantes dans la colonne 'fd_id'


## 🏷️: fr_id

In [155]:
display_stats(df_geo, "fr_id")

---

**Colonne: fr_id**

🔍 Type réel: object
📊 Mémoire: 1.97 MB
Longueur max du texte: 4
Valeurs uniques: 2
Valeurs les plus fréquentes:


,count
fr_id,
FR11,38824
FR12,92


✅ Pas de valeurs manquantes dans la colonne 'fr_id'


In [156]:
# Vérifie la cohérence entre fr_id et regrgp
df_geo.groupby("fr_id").agg({"regrgp_nom": "unique"})

,regrgp_nom
fr_id,
FR11,"[Province, Ile-de-France, DROM-COM]"
FR12,[DROM-COM]


In [157]:
# Pour FR11 combien de DROM-COM?
mask = (df_geo["fr_id"] == "FR11") & (df_geo["regrgp_nom"] == 'DROM-COM')
print(f"{df_geo.loc[mask, ].shape[0]} de lignes semblent incohérents")

152 de lignes semblent incohérents


## 🏷️: fe_id

In [158]:
display_stats(df_geo, "fe_id")

---

**Colonne: fe_id**

🔍 Type réel: object
📊 Mémoire: 1.93 MB
Longueur max du texte: 3
Valeurs uniques: 1
Valeurs les plus fréquentes:


,count
fe_id,
FE1,38916


✅ Pas de valeurs manquantes dans la colonne 'fe_id'


## 📍 Aire Urbaine

In [159]:
urban_area_cols = [col for col in df_geo.columns if 'au' in col]
print(f"Liste des colonnes d'aire urbaine:\n {urban_area_cols} \n {len(urban_area_cols)} colonnes")

Liste des colonnes d'aire urbaine:
 ['au_code', 'au_id', 'auc_id', 'auc_nom'] 
 4 colonnes


In [160]:
# Urban area columns for LUNEVILLE
mask = (df_geo["com_nom_maj"] == "LUNEVILLE")
df_geo.loc[mask, urban_area_cols]

,au_code,au_id,auc_id,auc_nom
21544,222,AU222,AU222,Lunéville


In [161]:
# Créer une série temporaire (n'affecte pas df_geo)
dep_code_numeric = pd.to_numeric(df_geo["dep_code"], errors='coerce')

# mask = (
#     (df_geo["com_nom"] != df_geo["auc_nom"]) &  (dep_code_numeric == 88) 
# )
# Commune avec aire urbaines SO (Sans Objet)
mask = ( df_geo["auc_nom"] == "SO" ) &  (dep_code_numeric == 88) 
df_geo.loc[mask, ["com_nom","auc_nom", "dep_code"]].head()


,com_nom,auc_nom,dep_code
36689,Aumontzey,SO,88
36705,Ban-sur-Meurthe,SO,88
36738,Le Boulay,SO,88
36743,Brancourt,SO,88
36783,Colroy-la-Grande,SO,88


In [162]:
verify_column_reproducibility(
    df_geo,
    "au_id",
    lambda df: "AU" + df["au_code"].astype('str')
)

❌ **au_id** ne peut PAS être complètement calculé à 100%<br>🟠 **32953 erreurs (84.7%)** sur 38916 lignes

,au_id,au_id_check
0,AU997,AUnan
1,AU002,AU2
2,SO,AUSO
3,AU002,AU2
4,AU002,AU2


### 🏷️: au_code

In [163]:
display_stats(df_geo, "au_code")

---

**Colonne: au_code**

🔍 Type réel: object
📊 Mémoire: 1.33 MB
Longueur max du texte: 5
Valeurs uniques: 798
Valeurs les plus fréquentes:


,count
au_code,
SO,3813
1,1771
2,507
4,452
25,290


⚠️ Il y a 17472 valeurs manquantes dans la colonne 'au_code' soit 44.9%


### 🏷️: au_id

In [164]:
display_stats(df_geo, "au_id")

---

**Colonne: au_id**

🔍 Type réel: object
📊 Mémoire: 1.99 MB
Longueur max du texte: 7
Valeurs uniques: 801
Valeurs les plus fréquentes:


,count
au_id,
AU000,7007
AU998,6604
SO,3898
AU997,3776
AU001,1771


✅ Pas de valeurs manquantes dans la colonne 'au_id'


### 🏷️: auc_id

In [165]:
display_stats(df_geo, "auc_id")

---

**Colonne: auc_id**

🔍 Type réel: object
📊 Mémoire: 2.01 MB
Longueur max du texte: 7
Valeurs uniques: 18185
Valeurs les plus fréquentes:


,count
auc_id,
SO,3898
AU001,1771
AU002,507
AU004,452
AU025,290


✅ Pas de valeurs manquantes dans la colonne 'auc_id'


### 🏷️: auc_nom

In [166]:
display_stats(df_geo, "auc_nom")

---

**Colonne: auc_nom**

🔍 Type réel: object
📊 Mémoire: 2.25 MB
Longueur max du texte: 58
Valeurs uniques: 17523
Valeurs les plus fréquentes:


,count
auc_nom,
SO,3813
Paris,1771
Lyon,507
Toulouse,452
Dijon,290


✅ Pas de valeurs manquantes dans la colonne 'auc_nom'


## 🏷️: geolocalisation

In [167]:
display_stats(df_geo, "geolocalisation")

---

**Colonne: geolocalisation**

🔍 Type réel: object
📊 Mémoire: 2.74 MB
Longueur max du texte: 32
Valeurs uniques: 36745
Valeurs les plus fréquentes:


,count
geolocalisation,
"46.1534255214,4.92611354223",1
"46.0091878776,5.42801696363",1
"45.9608475114,5.3729257777",1
"45.9961799872,4.91227250796",1
"45.7494989044,5.59432017366",1


⚠️ Il y a 2171 valeurs manquantes dans la colonne 'geolocalisation' soit 5.6000000000000005%


## 🔎RECHERCHE DE CLE CANDIDATE

In [168]:
_ = get_possible_candidates(df_geo, verbose=True)

#### 🔍 Analyse des colonnes pour trouver des clés candidates

> **🎯 3 clé candidate(s) trouvée(s) :**
>
> - ✅ `com_code`
> - ✅ `com_id`
> - ✅ `calc_CODCOM_INSEE`


# 🏙️ DONNEES COMMUNES (34 991, 9)

In [169]:
df_communes.columns

Index(['CODREG', 'CODDEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'PMUN', 'PCAP', 'PTOT'], dtype='object')

## 🏷️: CODREG

In [170]:
df_communes['CODREG'] = df_communes['CODREG'].astype('str')
display_stats(df_communes, 'CODREG')

---

**Colonne: CODREG**

🔍 Type réel: object
📊 Mémoire: 1.70 MB
Longueur max du texte: 2
Valeurs uniques: 17
Valeurs les plus fréquentes:


,count
CODREG,
44,5121
76,4454
75,4313
84,4038
32,3789


✅ Pas de valeurs manquantes dans la colonne 'CODREG'


## 🏷️: CODDEP

In [171]:
col = "CODDEP"
df_communes[col] = df_communes[col].apply(lambda x: to_padded_str(x, zero_pad=2))
display_stats(df_communes, col)

---

**Colonne: CODDEP**

🔍 Type réel: object
📊 Mémoire: 1.70 MB
Longueur max du texte: 3
Valeurs uniques: 100
Valeurs les plus fréquentes:


,count
CODDEP,
62,890
02,800
80,772
57,725
76,708


✅ Pas de valeurs manquantes dans la colonne 'CODDEP'


In [172]:
mask = df_communes["CODDEP"].str.len() == 3
df_communes.loc[mask, ]

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT
34878,1,971,2.0,93,101,Les Abymes,53514,513,54027
34879,1,971,2.0,14,102,Anse-Bertrand,4001,64,4065
34880,1,971,1.0,94,103,Baie-Mahault,30837,498,31335
34881,1,971,1.0,21,104,Baillif,5203,92,5295
34882,1,971,1.0,06,105,Basse-Terre,9861,244,10105
...,...,...,...,...,...,...,...,...,...
34986,4,974,1.0,04,420,Sainte-Suzanne,24065,227,24292
34987,4,974,3.0,06,421,Salazie,7136,73,7209
34988,4,974,2.0,99,422,Le Tampon,79824,1009,80833
34989,4,974,4.0,14,423,Les Trois-Bassins,7015,91,7106


## 🏷️: CODARR

In [173]:
df_communes['CODARR'] = df_communes['CODARR'].apply(lambda x: to_padded_str(x, zero_pad=2))
display_stats(df_communes,'CODARR')

---

**Colonne: CODARR**

🔍 Type réel: object
📊 Mémoire: 1.70 MB
Longueur max du texte: 2
Valeurs uniques: 9
Valeurs les plus fréquentes:


,count
CODARR,
02,10778
01,10592
03,8763
04,2932
05,1284


⚠️ Il y a 1 valeurs manquantes dans la colonne 'CODARR' soit 0.0%


In [174]:
# Assurer de ne pas avoir "nan" en format texte (len('nan') = 3)
mask = (df_communes['CODARR'].str.len() == 3)
df_communes.loc[mask,]

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT


## 🏷️: CODCAN

In [175]:
col = "CODCAN"
df_communes[col] = df_communes[col].apply(lambda x: to_padded_str(x, zero_pad=2))
display_stats(df_communes, col)

---

**Colonne: CODCAN**

🔍 Type réel: object
📊 Mémoire: 1.70 MB
Longueur max du texte: 2
Valeurs uniques: 59
Valeurs les plus fréquentes:


,count
CODCAN,
02,1885
01,1871
08,1820
10,1810
09,1735


⚠️ Il y a 1 valeurs manquantes dans la colonne 'CODCAN' soit 0.0%


In [176]:
df_communes[df_communes["CODCAN"] == '01']

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT
1,84,01,01,01,2,L'Abergement-de-Varey,256,1,257
2,84,01,01,01,4,Ambérieu-en-Bugey,14134,380,14514
5,84,01,01,01,7,Ambronay,2800,115,2915
6,84,01,01,01,8,Ambutrix,762,15,777
11,84,01,01,01,13,Arandas,143,4,147
...,...,...,...,...,...,...,...,...,...
34648,11,94,01,01,2,Alfortville,44805,161,44966
34844,11,95,02,01,555,Saint-Gratien,20895,166,21061
34850,11,95,01,01,582,Sannois,26565,336,26901
34967,4,974,02,01,401,Les Avirons,11440,210,11650


## 🏷️: CODCOM

In [177]:
col = "CODCOM"
df_communes[col] = df_communes[col].apply(lambda x: to_padded_str(x, zero_pad=3))
display_stats(df_communes, col)

---

**Colonne: CODCOM**

🔍 Type réel: object
📊 Mémoire: 1.74 MB
Longueur max du texte: 3
Valeurs uniques: 908
Valeurs les plus fréquentes:


,count
CODCOM,
077,89
104,89
046,88
061,88
070,87


✅ Pas de valeurs manquantes dans la colonne 'CODCOM'


## 🔑: calc_CODCOM_INSEE

In [178]:
# Recherche d'une clé candidate
df_communes["calc_CODCOM_INSEE"] = df_communes["CODDEP"] + df_communes["CODCOM"]

tmp_len_cod_com = df_communes["calc_CODCOM_INSEE"].astype('str').str.len().max()
if tmp_len_cod_com != 5:
    display(Markdown(f"⚠️ **Attention** : La longuer du code commune INSEE doît être 5! et non pas de {tmp_len_cod_com}"))

⚠️ **Attention** : La longuer du code commune INSEE doît être 5! et non pas de 6

In [179]:
mask = (df_communes["calc_CODCOM_INSEE"].str.len() == 6)
df_communes.loc[mask, "calc_CODCOM_INSEE"].shape[0]

113

In [180]:
mask = df_communes["CODCOM"].str.len() == df_communes["CODDEP"].str.len()
df_commune_cc_a_6_chiffres = df_communes.loc[mask, ]
df_commune_cc_a_6_chiffres.head()

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT,calc_CODCOM_INSEE
34878,1,971,02,93,101,Les Abymes,53514,513,54027,971101
34879,1,971,02,14,102,Anse-Bertrand,4001,64,4065,971102
34880,1,971,01,94,103,Baie-Mahault,30837,498,31335,971103
34881,1,971,01,21,104,Baillif,5203,92,5295,971104
34882,1,971,01,06,105,Basse-Terre,9861,244,10105,971105


#### Correction des code commune INSEE à 6 chiffre en supprimant un chiffre excedentaire de code commune

In [181]:
df_commune_cc_a_6_chiffres.loc[:,'calc_CODCOM_INSEE'] = df_commune_cc_a_6_chiffres.apply(
                            lambda row: (
                                str(row['CODDEP'])[:2] if len(str(row['CODDEP'])) == 3 
                            else str(row['CODDEP']).zfill(2)) + str(row['CODCOM']).zfill(3), 
                            axis=1
)

In [182]:
df_commune_cc_a_6_chiffres.to_csv("./data/extraits/corrections_code_commune_df_commune.csv", sep=";")

In [183]:
tmp_cols = ["com_nom","COM","calc_CODCOM_INSEE"]
tmp_merge = df_commune_cc_a_6_chiffres.merge(df_geo, how='inner', on='calc_CODCOM_INSEE')[tmp_cols]
tmp_merge.loc[tmp_merge["COM"] != tmp_merge["com_nom"], tmp_cols]

,com_nom,COM,calc_CODCOM_INSEE
63,Les Trois-Ilets,Les Trois-ÃŽlets,97231
93,Petite-Ile,Petite-ÃŽle,97405


In [184]:
# Mets les lignes corrigées dans la dataframe original
shape_before = df_communes.shape

df_communes.loc[df_commune_cc_a_6_chiffres.index,:] = df_commune_cc_a_6_chiffres

report_shape_changes(shape_before= shape_before, shape_after=df_communes.shape)

Shape avant: (34991, 10)
Shape après: (34991, 10)
  🔄 Aucun changement de dimension


## 🏷️: COM

In [185]:
col = "COM"
df_communes[col] = df_communes[col].apply(lambda x: to_padded_str(x, zero_pad=3))
display_stats(df_communes, col)

---

**Colonne: COM**

🔍 Type réel: object
📊 Mémoire: 2.14 MB
Longueur max du texte: 45
Valeurs uniques: 32732
Valeurs les plus fréquentes:


,count
COM,
Sainte-Colombe,12
Saint-Sauveur,11
Beaulieu,10
Saint-Aubin,10
Saint-Loup,9


✅ Pas de valeurs manquantes dans la colonne 'COM'


In [186]:
df_communes.loc[df_communes["COM"].str.len().idxmax(),"COM"]

'Saint-Remy-en-Bouzemont-Saint-Genest-et-Isson'

In [187]:
mask = df_communes["COM"] == "Lunéville"
df_communes.loc[mask,]

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT,calc_CODCOM_INSEE
19772,44,54,02,98,329,Lunéville,17867,324,18191,54329


## 🏷️: PMUN

In [188]:
col = "PMUN"
df_communes[col] = df_communes[col].astype(float)
display_stats(df_communes, col)

---

**Colonne: PMUN**

🔍 Type réel: float64
⚠️ Cette colonne pourrait être convertie en int
📊 Mémoire: 0.27 MB


,PMUN
count,"34_991,00"
mean,"1_915,38"
std,"8_679,35"
min,"0,00"
25%,"198,00"
50%,"457,00"
75%,"1_162,00"
max,"493_465,00"


✅ Pas de valeurs manquantes dans la colonne 'PMUN'


## 🏷️: PCAP

In [189]:
col = "PCAP"
display_stats(df_communes, col)

---

**Colonne: PCAP**

🔍 Type réel: int64
📊 Mémoire: 0.27 MB


,PCAP
count,"34_991,00"
mean,"35,48"
std,"132,33"
min,"0,00"
25%,"4,00"
50%,"9,00"
75%,"24,00"
max,"5_167,00"


✅ Pas de valeurs manquantes dans la colonne 'PCAP'


## 🏷️: PTOT

In [190]:
col = "PTOT"
display_stats(df_communes, col)

---

**Colonne: PTOT**

🔍 Type réel: int64
📊 Mémoire: 0.27 MB


,PTOT
count,"34_991,00"
mean,"1_950,85"
std,"8_791,06"
min,"0,00"
25%,"202,00"
50%,"468,00"
75%,"1_189,00"
max,"498_596,00"


✅ Pas de valeurs manquantes dans la colonne 'PTOT'


### PTOT = PMUN + PCAP?

In [191]:
verify_column_reproducibility(
    df_communes,
    "PTOT",
    lambda df: df["PMUN"] + df["PCAP"]    
)

✅ **PTOT** peut être calculé à 100%

## 🔎RECHERCHE DE CLE CANDIDATE

In [192]:
__ = get_possible_candidates(df_communes, verbose=True)

#### 🔍 Analyse des colonnes pour trouver des clés candidates

> **🎯 1 clé candidate(s) trouvée(s) :**
>
> - ✅ `calc_CODCOM_INSEE`


# 🚌 Migration des données vers la base SQL

## 🆎Préparation à la migration

In [193]:
from sqlalchemy import create_engine
import sys,os
from dotenv import load_dotenv

In [ ]:
load_dotenv()  # charge .env depuis le répertoire courant
server   = os.environ.get('DB_HOST', 'localhost\\SQLEXPRESS')
database = os.environ.get('DB_NAME', 'LaplaceImmo')
driver   = os.environ.get('DB_DRIVER', 'ODBC Driver 17 for SQL Server')
password = os.environ.get('DB_PASSWORD', '')

connection_string = (
    f"mssql+pyodbc://SA:{password}@{server}/{database}"
    f"?driver={driver}&TrustServerCertificate=yes"
)

Databasename: LaplaceImmo


In [195]:
# Création du "moteur" de connexion à la base de données
try:
    engine = create_engine(connection_string)
    print("Connexion à la base de données réussie !")
except Exception as e:
    print(f"Erreur de connexion à la base de données : {e}")
    sys.exit() # Arrête le script si la connexion échoue

Connexion à la base de données réussie !


## 💾 Variable à utiliser pour la base SQL

In [196]:
join_key = "calc_CODCOM_INSEE"
population_total = "PTOT"
nom_commune = "Commune"
type_local = "Type local"
date_mutation = "Date mutation"
reg_nom = "reg_nom"
surface_carrez = "Surface Carrez du 1er lot"
surface_local = "Surface reelle bati"
valeur_fonciere = "Valeur fonciere"
total_piece = "Nombre pieces principales"
cod_dep = "Code departement"

In [197]:
print(f"df_geo est unique ? : {'OUI ✅' if df_geo[join_key].nunique() == df_geo.shape[0] else 'NON ❌'}")
print(f"df_communes est unique ? : {'OUI ✅' if df_communes[join_key].nunique() == df_communes.shape[0] else 'NON ❌'}")
print(f"df_fonciere est unique ? : {'OUI ✅' if df_fonciere[join_key].nunique() == df_fonciere.shape[0] else 'NON ❌'}")

df_geo est unique ? : OUI ✅
df_communes est unique ? : OUI ✅
df_fonciere est unique ? : NON ❌


## 📦 Création de la table `LaplaceImmo.commune`

In [198]:
# --- On fait des copies de travail une bonne fois pour toutes ---
df_communes_work = df_communes.copy()
df_geo_work = df_geo.copy()
df_fonciere_work = df_fonciere.copy()

# Créer les tables de références
print_md("**Création de geo_ref ...**")
shape_before = df_geo_work.shape
geo_ref = df_geo_work.drop_duplicates(subset=[join_key])[[join_key, reg_nom]]
report_shape_changes(shape_before, geo_ref.shape)

print_md("**Création de fonciere_ref ...**")
shape_before = df_fonciere_work.shape
fonciere_ref = df_fonciere_work.drop_duplicates(subset=[join_key], keep='first')[[join_key, 'Code postal']]
report_shape_changes(shape_before, fonciere_ref.shape)


print_md("**Création de communes_ref ...**")
shape_before = df_communes_work.shape
communes_ref = df_communes_work.drop_duplicates(subset=[join_key])
report_shape_changes(shape_before, communes_ref.shape)

**Création de geo_ref ...**

Shape avant: (38916, 38)
Shape après: (38916, 2)
  🗑️  Colonnes supprimées: 36


**Création de fonciere_ref ...**

Shape avant: (34169, 47)
Shape après: (3125, 2)
  ✂️  Lignes supprimées: 31044
  🗑️  Colonnes supprimées: 45


**Création de communes_ref ...**

Shape avant: (34991, 10)
Shape après: (34991, 10)
  🔄 Aucun changement de dimension


### Construction de dataframe qui va se servir pour populer la table **commune**

In [199]:
# --- On construit notre DataFrame final en enchaînant les opérations ---

print_md("**Création de commune_df_final en ajoutant le nom de la region ...**")
shape_before = communes_ref.shape
commune_df_final = pd.merge(communes_ref, geo_ref, on=join_key, how='left')
report_shape_changes(shape_before, commune_df_final.shape)

print_md("**Ajout de code postal à la commune_df_final ...**")
shape_before = commune_df_final.shape
commune_df_final = pd.merge(commune_df_final, fonciere_ref, on=join_key, how='left')
report_shape_changes(shape_before, commune_df_final.shape)

**Création de commune_df_final en ajoutant le nom de la region ...**

Shape avant: (34991, 10)
Shape après: (34991, 11)
  📊 Colonnes ajoutées: 1


**Ajout de code postal à la commune_df_final ...**

Shape avant: (34991, 11)
Shape après: (34991, 12)
  📊 Colonnes ajoutées: 1


In [200]:
print_md("**Mapping commune_df_final ...**")
shape_before = commune_df_final.shape
commune_df_final = commune_df_final[[
    join_key, 'CODDEP', 'CODCOM', 'COM', 'Code postal', 'PTOT', reg_nom
]].rename(columns={
    join_key: 'code_insee',
    'CODDEP': 'code_departement',
    'CODCOM': 'code_commune',
    'COM': 'nom_commune',
    'Code postal': 'code_postal',
    'PTOT': 'population_total',
    reg_nom: 'nom_region'
})
report_shape_changes(shape_before, commune_df_final.shape)

**Mapping commune_df_final ...**

Shape avant: (34991, 12)
Shape après: (34991, 7)
  🗑️  Colonnes supprimées: 5


In [201]:
# --- Nettoyage final des types et des valeurs manquantes ---
commune_df_final['population_total'] = pd.to_numeric(commune_df_final['population_total'], errors='coerce').fillna(0).astype(int)

print_md("**Informations sur le DataFrame final:**")
commune_df_final.info()

if commune_df_final['code_insee'].is_unique:
    print(f"\n✅ La clé primaire 'code_insee' est bien unique. Le DataFrame contient {len(commune_df_final)} lignes prêtes à être insérées.")
else:
    print(f"\n❌ ATTENTION : La clé primaire 'code_insee' contient des doublons.")

**Informations sur le DataFrame final:**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34991 entries, 0 to 34990
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   code_insee        34991 non-null  object
 1   code_departement  34991 non-null  object
 2   code_commune      34991 non-null  object
 3   nom_commune       34991 non-null  object
 4   code_postal       3125 non-null   object
 5   population_total  34991 non-null  int64 
 6   nom_region        34991 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.9+ MB

✅ La clé primaire 'code_insee' est bien unique. Le DataFrame contient 34991 lignes prêtes à être insérées.


In [202]:
commune_df_final.head()

,code_insee,code_departement,code_commune,nom_commune,code_postal,population_total,nom_region
0,01001,01,001,L'Abergement-Clémenciat,NaN,798,Auvergne-Rhône-Alpes
1,01002,01,002,L'Abergement-de-Varey,NaN,257,Auvergne-Rhône-Alpes
2,01004,01,004,Ambérieu-en-Bugey,01500,14514,Auvergne-Rhône-Alpes
3,01005,01,005,Ambérieux-en-Dombes,NaN,1776,Auvergne-Rhône-Alpes
4,01006,01,006,Ambléon,NaN,118,Auvergne-Rhône-Alpes


### Population de commune_df_final vers la table **commune**

In [203]:
# Remise à zéro des tables — ordre FK obligatoire (enfant avant parent)
with engine.connect() as connection:
    connection.execute(text("DELETE FROM dbo.vente"))
    connection.execute(text("DELETE FROM dbo.bien"))
    connection.execute(text("DELETE FROM dbo.commune"))
    connection.commit()
print("✅ Tables vidées — prêt pour l'insertion")

✅ Tables vidées — prêt pour l'insertion


In [204]:
try:
    commune_df_final.to_sql('commune', con=engine, if_exists='append', index=False)
    print(f"✅ {len(commune_df_final)} lignes insérées dans 'commune'.")
except Exception as e:
    print(f"❌ ERREUR : {e}")

✅ 34991 lignes insérées dans 'commune'.


## Préparation de dataframe df_fonciere

In [205]:
df_fonciere_work[['Valeur fonciere','Surface Carrez du 1er lot','Surface reelle bati']].describe()

,Valeur fonciere,Surface Carrez du 1er lot,Surface reelle bati
count,3.415100e+04,34169.000000,34169.000000
mean,2.528471e+05,57.644635,56.724458
std,3.252594e+05,53.335654,29.881797
min,5.375000e+02,0.400000,1.000000
25%,1.040000e+05,34.600000,35.000000
50%,1.690000e+05,53.000000,53.000000
75%,2.850000e+05,72.310000,72.000000
max,9.000000e+06,5153.000000,379.000000


In [206]:
# Création de l'identifiant unique de transaction (qui servira de clé primaire)
# On le base sur l'index pour garantir l'unicité
df_fonciere_work.reset_index(drop=True, inplace=True)
df_fonciere_work['id_transaction'] = df_fonciere_work.index + 1 # +1 pour que les ID commencent à 1


# --- Vérification finale ---
print("\nDataFrame 'df_fonciere_work' préparé.")
display(df_fonciere_work[[join_key, 'id_transaction', 'Type local', 'Valeur fonciere']].head())
# df_fonciere_work.info()


DataFrame 'df_fonciere_work' préparé.


,calc_CODCOM_INSEE,id_transaction,Type local,Valeur fonciere
0,01103,1,Appartement,165000.0
1,06004,2,Appartement,355680.0
2,06088,3,Appartement,229500.0
3,06123,4,Appartement,125000.0
4,13005,5,Appartement,90000.0


## 📦 Création de la table `vente` et `bien`

In [207]:
# --- Création du DataFrame `bien_df` ---
shape_before = df_fonciere_work.shape

# Mapping
bien_df = df_fonciere_work.rename(columns={
    'id_transaction': 'id_bien',
    'No voie': 'no_voie',
    'B/T/Q': 'btq',
    'Voie': 'voie',
    'Type de voie': 'type_voie',
    'Nombre pieces principales': 'total_piece',
    'Surface Carrez du 1er lot': 'surface_carrez',
    'Surface reelle bati': 'surface_local',
    'Type local': 'type_local',
    join_key: 'code_insee'
})

# Fitrer que les colonnes nécessaires
bien_df = bien_df[[
    'id_bien', 'no_voie', 'btq', 'voie', 'type_voie', 'total_piece', 
    'surface_carrez', 'surface_local', 'type_local', 'code_insee'
]]


def column_empty_status(df, column_name):
    if df[column_name].isna().any():
        return f"❌ '{column_name}' contient des valeurs vides."
    else:
        return f"✅ '{column_name}' est complètement remplie."


for col in ['surface_carrez', 'surface_local', 'total_piece']:
    display(column_empty_status(bien_df, col))

# Finalisation : Remplacer les NaN potentiels pour les colonnes qui sont NOT NULL en BDD
# bien_df[['surface_carrez', 'surface_local']] = bien_df[['surface_carrez', 'surface_local']].fillna(0)
# bien_df['total_piece'] = bien_df['total_piece'].fillna(0).astype(int)

report_shape_changes(shape_before, bien_df.shape)

"✅ 'surface_carrez' est complètement remplie."

"✅ 'surface_local' est complètement remplie."

"✅ 'total_piece' est complètement remplie."

Shape avant: (34169, 48)
Shape après: (34169, 10)
  🗑️  Colonnes supprimées: 38


In [208]:
# --- Création du DataFrame `vente_df` ---
print_md("**Création du DataFrame 'vente_df' pour la table 'vente'...**")
shape_before = df_fonciere_work.shape

# On sélectionne les colonnes nécessaires et on les renomme
vente_df = df_fonciere_work.rename(columns={
    'id_transaction': 'id_vente',
    'Date mutation': 'date_mutation',
    'Valeur fonciere': 'valeur_fonciere'
})

# On crée la clé étrangère pour lier la vente au bien
vente_df['id_bien'] = vente_df['id_vente']

# On ne garde que les colonnes finales dans le bon ordre
vente_df = vente_df[['id_vente', 'date_mutation', 'valeur_fonciere', 'id_bien']]

report_shape_changes(shape_before, vente_df.shape)

**Création du DataFrame 'vente_df' pour la table 'vente'...**

Shape avant: (34169, 48)
Shape après: (34169, 4)
  🗑️  Colonnes supprimées: 44


**--- VÉRIFICATION DES DATAFRAMES FINALS ---**


In [209]:
print_md("**Aperçu et informations sur `bien_df`:**")
display(bien_df.head())
bien_df.info()

**Aperçu et informations sur `bien_df`:**

,id_bien,no_voie,btq,voie,type_voie,total_piece,surface_carrez,surface_local,type_local,code_insee
0,1,347,NaN,DU CHATEAU,RUE,3,48.22,48,Appartement,01103
1,2,4,NaN,EDOUARD BAUDOIN,BD,1,39.11,40,Appartement,06004
2,3,20,B,MARCEAU,RUE,3,80.25,82,Appartement,06088
3,4,550,NaN,DES VESPINS RN7,RTE,1,27.51,27,Appartement,06123
4,5,9300,NaN,LES ARPEGES BD DES ABA,RES,2,47.33,47,Appartement,13005


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34169 entries, 0 to 34168
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_bien         34169 non-null  int64  
 1   no_voie         34036 non-null  Int64  
 2   btq             2174 non-null   object 
 3   voie            34169 non-null  object 
 4   type_voie       33229 non-null  object 
 5   total_piece     34169 non-null  int64  
 6   surface_carrez  34169 non-null  float64
 7   surface_local   34169 non-null  int64  
 8   type_local      34169 non-null  object 
 9   code_insee      34169 non-null  object 
dtypes: Int64(1), float64(1), int64(3), object(5)
memory usage: 2.6+ MB


In [210]:
print_md("\n**Aperçu et informations sur `vente_df`:**")
display(vente_df.head())
vente_df.info()


**Aperçu et informations sur `vente_df`:**

,id_vente,date_mutation,valeur_fonciere,id_bien
0,1,2020-01-02,165000.0,1
1,2,2020-01-02,355680.0,2
2,3,2020-01-02,229500.0,3
3,4,2020-01-02,125000.0,4
4,5,2020-01-02,90000.0,5


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34169 entries, 0 to 34168
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_vente         34169 non-null  int64         
 1   date_mutation    34169 non-null  datetime64[ns]
 2   valeur_fonciere  34151 non-null  float64       
 3   id_bien          34169 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 1.0 MB


In [211]:
df_fonciere_work["B/T/Q"].value_counts().head()

B/T/Q
B    1424
A     219
T     186
F     149
C      72
Name: count, dtype: int64

In [212]:
# --- Étape de nettoyage FINAL avant insertion ---
# On s'assure que les NaN sont bien gérés (c'est une sécurité)
bien_df['type_voie'] = bien_df['type_voie'].replace(np.nan, None)
bien_df['btq'] = bien_df['btq'].replace(np.nan, None)

In [213]:
def sql_insert_bien():
    print("\n--- Tentative d'insertion par blocs dans la table 'bien' (avec COMMIT explicite) ---")
    try:
        
        # --- Définition des blocs ---
        chunk_size = 1000
        total_lignes = len(bien_df)
        nombre_de_blocs = math.ceil(total_lignes / chunk_size)
        print(f"Préparation à l'insertion de {total_lignes} lignes en {nombre_de_blocs} blocs.")

        # --- La boucle d'insertion ---
        with engine.connect() as connection:
            # 1. On démarre une transaction manuellement
            trans = connection.begin()
        
            try:
                # 2. On active IDENTITY_INSERT à l'intérieur de la transaction
                connection.execute(text('SET IDENTITY_INSERT bien ON;'))
            
                for i in range(nombre_de_blocs):
                    start = i * chunk_size
                    end = start + chunk_size
                    print(f"  -> Insertion du bloc {i + 1}/{nombre_de_blocs}...")
                    df_chunk = bien_df.iloc[start:end]
                    df_chunk.to_sql('bien', con=connection, if_exists='append', index=False)
            
                # 3. On désactive IDENTITY_INSERT
                connection.execute(text('SET IDENTITY_INSERT bien OFF;'))
            
                # 4. C'EST L'ORDRE LE PLUS IMPORTANT : ON VALIDE LA TRANSACTION
                trans.commit()
            
                print(f"\n✅ Transaction validée avec succès. {total_lignes} lignes devraient être dans la table 'bien'.")

            except Exception as inner_e:
                print(f"\n❌ ERREUR DANS LA TRANSACTION, ANNULATION... : {inner_e}")
                # En cas d'erreur, on annule tout
                trans.rollback()
                raise # On relance l'erreur pour voir ce qui s'est passé

    except Exception as outer_e:
        print(f"\n❌ ERREUR LORS DE LA CONNEXION OU DE L'INSERTION : {outer_e}")


sql_insert_bien()


--- Tentative d'insertion par blocs dans la table 'bien' (avec COMMIT explicite) ---
Préparation à l'insertion de 34169 lignes en 35 blocs.
  -> Insertion du bloc 1/35...
  -> Insertion du bloc 2/35...
  -> Insertion du bloc 3/35...
  -> Insertion du bloc 4/35...
  -> Insertion du bloc 5/35...
  -> Insertion du bloc 6/35...
  -> Insertion du bloc 7/35...
  -> Insertion du bloc 8/35...
  -> Insertion du bloc 9/35...
  -> Insertion du bloc 10/35...
  -> Insertion du bloc 11/35...
  -> Insertion du bloc 12/35...
  -> Insertion du bloc 13/35...
  -> Insertion du bloc 14/35...
  -> Insertion du bloc 15/35...
  -> Insertion du bloc 16/35...
  -> Insertion du bloc 17/35...
  -> Insertion du bloc 18/35...
  -> Insertion du bloc 19/35...
  -> Insertion du bloc 20/35...
  -> Insertion du bloc 21/35...
  -> Insertion du bloc 22/35...
  -> Insertion du bloc 23/35...
  -> Insertion du bloc 24/35...
  -> Insertion du bloc 25/35...
  -> Insertion du bloc 26/35...
  -> Insertion du bloc 27/35...
  ->

In [214]:
def sql_insert_vente():
# ===================================================================
# CELLULE D'INSERTION POUR LA TABLE 'vente' 
# ===================================================================

    print("\n--- Tentative d'insertion dans la table 'vente' (avec COMMIT explicite) ---")
    try:
        with engine.connect() as connection:
            # 1. On démarre une transaction manuellement
            trans = connection.begin()
        
            try:
                # 2. On active IDENTITY_INSERT
                connection.execute(text('SET IDENTITY_INSERT vente ON;'))
            
                # 3. Insertion des données
                print(f"Insertion de {len(vente_df)} lignes...")
                vente_df.to_sql(
                    'vente', 
                    con=connection, 
                    if_exists='append',
                    index=False
                )
            
                # 4. On désactive IDENTITY_INSERT
                connection.execute(text('SET IDENTITY_INSERT vente OFF;'))
            
                # 5. ON VALIDE LA TRANSACTION
                trans.commit()
            
                print(f"\n✅ Transaction validée avec succès. {len(vente_df)} lignes insérées dans la table 'vente'.")

            except Exception as inner_e:
                print(f"\n❌ ERREUR DANS LA TRANSACTION, ANNULATION... : {inner_e}")
                trans.rollback()
                raise

    except Exception as outer_e:
        print(f"\n❌ ERREUR LORS DE LA CONNEXION OU DE L'INSERTION : {outer_e}")

sql_insert_vente()


--- Tentative d'insertion dans la table 'vente' (avec COMMIT explicite) ---
Insertion de 34169 lignes...

✅ Transaction validée avec succès. 34169 lignes insérées dans la table 'vente'.
